In [3]:
# Cell 1: install deps
!pip install torch transformers accelerate peft bitsandbytes datasets scikit-learn pandas numpy tqdm joblib pyarrow huggingface_hub
!pip install transformers accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
^C
ERROR: Operation cancelled by user
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


In [4]:
!rm -rf /kaggle/working/*

In [6]:
"""
datafetching.py — Download MCQ datasets and export as CSV matching the
pipeline's expected format:

    article, question, A, B, C, D, answer

Datasets:
  RACE          1. Hugging Face (ehovy/race)  2. Kaggle (ankitdhiman7/race-dataset)
  DREAM         1. GitHub (nlpdata/dream)     2. — (HF loading scripts no longer supported)
  CommonsenseQA 1. Hugging Face (commonsense_qa)
  ARC           1. Hugging Face (allenai/ai2_arc)
  SocialIQA     1. Google Storage              (HF loading scripts no longer supported)
  MultiRC       1. super_glue (Parquet)
  OpenBookQA    1. Hugging Face (allenai/openbookqa)
  SciQ          1. Hugging Face (sciq Parquet)
  MMLU          1. Hugging Face (cais/mmlu Parquet)
  MedQA         1. Hugging Face (GBaker/MedQA-USMLE-4-options-hf JSONL)
  QASC          1. Hugging Face (qasc Parquet, 8→4 options)
"""

from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path
from typing import List, Optional

import pandas as pd

# ---------------------------------------------------------------------------
# Paths — works in Kaggle/Colab notebooks and local scripts
# ---------------------------------------------------------------------------

try:
    _THIS_DIR = Path(__file__).resolve().parent
except NameError:
    _THIS_DIR = Path(os.getcwd())
DATA_RAW = _THIS_DIR / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _pad_options(opts: List[str], n: int = 4) -> List[str]:
    while len(opts) < n:
        opts.append("(none of the above)")
    return opts[:n]


def _answer_letter(idx: int) -> str:
    return chr(65 + idx)


# ---------------------------------------------------------------------------
# 1. RACE
# ---------------------------------------------------------------------------

def _race_from_hf() -> Optional[pd.DataFrame]:
    """Try loading RACE via Hugging Face datasets library."""
    try:
        from datasets import load_dataset
    except ImportError:
        return None

    print("[RACE] trying HuggingFace ehovy/race (config='all') …")
    try:
        ds = load_dataset("ehovy/race", "all")
    except Exception as e:
        print(f"  HF failed: {e}")
        return None

    rows = []
    for split_name in ds.keys():
        for ex in ds[split_name]:
            opts = ex["options"]
            rows.append({
                "article": ex["article"],
                "question": ex["question"],
                "A": opts[0] if len(opts) > 0 else "",
                "B": opts[1] if len(opts) > 1 else "",
                "C": opts[2] if len(opts) > 2 else "",
                "D": opts[3] if len(opts) > 3 else "",
                "answer": str(ex["answer"]).strip().upper(),
            })
    return pd.DataFrame(rows)


def _race_from_kaggle() -> Optional[pd.DataFrame]:
    """Fallback: download RACE CSV from Kaggle."""
    print("[RACE] trying Kaggle ankitdhiman7/race-dataset …")
    dest = DATA_RAW / "kaggle_race"
    dest.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        ["kaggle", "datasets", "download", "ankitdhiman7/race-dataset", "-p", str(dest)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"  Kaggle API failed. Download manually from:")
        print("    https://www.kaggle.com/datasets/ankitdhiman7/race-dataset")
        print(f"  Error: {result.stderr[:200]}")
        return None

    zips = list(dest.glob("*.zip"))
    if not zips:
        return None
    with zipfile.ZipFile(zips[0], "r") as zf:
        zf.extractall(dest)

    csvs = list(dest.rglob("*.csv")) + list(dest.rglob("*.CSV"))
    if not csvs:
        return None
    df = pd.read_csv(csvs[0])
    required = {"article", "question", "A", "B", "C", "D", "answer"}
    if not required.issubset(df.columns):
        print(f"  CSV missing columns: {required - set(df.columns)}")
        return None
    return df


def fetch_race() -> pd.DataFrame:
    df = _race_from_hf()
    if df is not None and len(df) > 0:
        print(f"[RACE]   → {len(df):,} rows (HuggingFace)")
        return df
    df = _race_from_kaggle()
    if df is not None and len(df) > 0:
        print(f"[RACE]   → {len(df):,} rows (Kaggle)")
        return df
    print("[RACE]   WARNING: no data loaded")
    return pd.DataFrame()


# ---------------------------------------------------------------------------
# 2. DREAM  (GitHub — nlpdata/dream)
# ---------------------------------------------------------------------------

def fetch_dream() -> pd.DataFrame:
    """
    Download DREAM from GitHub, parse JSON → flat DataFrame.
    DREAM has 3-option MCQs; we pad to 4 with placeholders.
    """
    url = "https://github.com/nlpdata/dream/archive/refs/heads/master.zip"
    zip_path = DATA_RAW / "dream-master.zip"
    extract_dir = DATA_RAW / "dream-master"

    print(f"[DREAM] downloading from GitHub …")
    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_RAW)

    all_rows = []
    for split_name in ("train", "dev", "test"):
        json_path = extract_dir / "data" / f"{split_name}.json"
        if not json_path.exists():
            continue
        with open(json_path) as f:
            data = json.load(f)
        for entry in data:
            dialogue = " ".join(str(t) for t in entry[0])
            for q in entry[1]:
                choices = q.get("choice", [])
                ans_text = q.get("answer", choices[0] if choices else "")
                opts = _pad_options(choices, 4)
                try:
                    ans_idx = choices.index(ans_text)
                except ValueError:
                    ans_idx = 0
                all_rows.append({
                    "article": dialogue,
                    "question": q.get("question", ""),
                    "A": opts[0],
                    "B": opts[1],
                    "C": opts[2],
                    "D": opts[3],
                    "answer": _answer_letter(ans_idx),
                })
    df = pd.DataFrame(all_rows)
    print(f"[DREAM]   → {len(df):,} rows (GitHub)")
    return df


# ---------------------------------------------------------------------------
# 3. CommonsenseQA  (HuggingFace — 5 options, no passage)
# ---------------------------------------------------------------------------

def fetch_commonsenseqa() -> pd.DataFrame:
    """Download CommonsenseQA from HuggingFace. 5 options → 4."""
    print("[CSQA] loading commonsense_qa …")
    try:
        from datasets import load_dataset
    except ImportError:
        print("  pip install datasets")
        return pd.DataFrame()

    try:
        ds = load_dataset("commonsense_qa")
    except Exception as e:
        print(f"  Failed: {e}")
        return pd.DataFrame()

    rows = []
    for split in ds:
        for ex in ds[split]:
            texts = list(ex["choices"]["text"])
            labels = list(ex["choices"]["label"])
            ans_key = str(ex["answerKey"]).strip().upper()
            try:
                ans_idx = labels.index(ans_key)
            except ValueError:
                ans_idx = 0
            if len(texts) > 4:
                if ans_idx >= 4:
                    texts[3], texts[ans_idx] = texts[ans_idx], texts[3]
                    ans_idx = 3
                opts = texts[:4]
            else:
                opts = _pad_options(texts, 4)
            rows.append({
                "article": ex["question"],
                "question": "",
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx),
            })
    df = pd.DataFrame(rows)
    print(f"[CSQA]   → {len(df):,} rows (HuggingFace)")
    return df


# ---------------------------------------------------------------------------
# 4. ARC (AI2 Reasoning Challenge)  — HuggingFace
# ---------------------------------------------------------------------------

def fetch_arc() -> pd.DataFrame:
    """Download ARC (Challenge + Easy) from HuggingFace."""
    print("[ARC] loading allenai/ai2_arc …")
    try:
        from datasets import load_dataset
    except ImportError:
        print("  pip install datasets")
        return pd.DataFrame()

    rows = []
    for config in ("ARC-Challenge", "ARC-Easy"):
        try:
            ds = load_dataset("allenai/ai2_arc", config)
        except Exception as e:
            print(f"  {config} failed: {e}")
            continue
        for split in ds:
            for ex in ds[split]:
                texts = list(ex["choices"]["text"])
                labels = list(ex["choices"]["label"])
                ans_raw = str(ex.get("answerKey", "")).strip()
                if not ans_raw:
                    continue
                try:
                    ans_idx = labels.index(ans_raw)
                except ValueError:
                    ans_idx = 0
                opts = _pad_options(texts, 4)
                if ans_idx >= len(opts):
                    ans_idx = 0
                rows.append({
                    "article": ex["question"],
                    "question": "",
                    "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                    "answer": _answer_letter(ans_idx),
                })
    df = pd.DataFrame(rows)
    print(f"[ARC]     → {len(df):,} rows (HuggingFace)")
    return df


# ---------------------------------------------------------------------------
# 5. SocialIQA  (HuggingFace — 3 options, has passage context)
# ---------------------------------------------------------------------------

def fetch_socialiqa() -> pd.DataFrame:
    """Download SocialIQA from Google Storage (HF loading script no longer supported)."""
    print("[SIQA] downloading from Google Storage …")
    url = "https://storage.googleapis.com/ai2-mosaic/public/socialiqa/socialiqa-train-dev.zip"
    zip_path = DATA_RAW / "socialiqa-train-dev.zip"
    extract_dir = DATA_RAW / "socialiqa-train-dev"

    try:
        urllib.request.urlretrieve(url, zip_path)
    except Exception as e:
        print(f"  Download failed: {e}")
        return pd.DataFrame()

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_RAW)

    rows = []
    for split_name in ("train", "dev"):
        jsonl_path = extract_dir / "socialiqa-train-dev" / f"{split_name}.jsonl"
        labels_path = extract_dir / "socialiqa-train-dev" / f"{split_name}-labels.lst"
        if not jsonl_path.exists() or not labels_path.exists():
            continue

        with open(labels_path) as f:
            labels = [line.strip() for line in f if line.strip()]

        with open(jsonl_path) as f:
            for i, line in enumerate(f):
                if i >= len(labels):
                    break
                ex = json.loads(line)
                ans_idx = int(labels[i]) - 1
                if ans_idx < 0:
                    ans_idx = 0
                opts = [ex["answerA"], ex["answerB"], ex["answerC"]]
                opts_padded = _pad_options(opts, 4)
                rows.append({
                    "article": ex["context"],
                    "question": ex.get("question", ""),
                    "A": opts_padded[0], "B": opts_padded[1],
                    "C": opts_padded[2], "D": opts_padded[3],
                    "answer": _answer_letter(ans_idx),
                })
    df = pd.DataFrame(rows)
    print(f"[SIQA]    → {len(df):,} rows (Google Storage)")
    return df


# ---------------------------------------------------------------------------
# 6. MultiRC  (HuggingFace — variable answers, multiple correct per Q)
# ---------------------------------------------------------------------------

def fetch_multirc() -> pd.DataFrame:
    """Download MultiRC from super_glue (Parquet). Pick first correct answer per Q."""
    print("[MultiRC] loading super_glue/multirc …")
    try:
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        print("  pip install huggingface_hub pyarrow")
        return pd.DataFrame()

    rows = []
    for split in ("train", "validation", "test"):
        try:
            path = hf_hub_download(
                repo_id="super_glue",
                filename=f"multirc/{split}-00000-of-00001.parquet",
                repo_type="dataset",
            )
        except Exception as e:
            print(f"  {split} download failed: {e}")
            continue

        table = pq.read_table(path)
        df_split = table.to_pandas()

        for (para, q_text), group in df_split.groupby(["paragraph", "question"], sort=False):
            answers = group["answer"].tolist()
            labels = group["label"].tolist()

            correct_texts = [
                a for a, lbl in zip(answers, labels) if lbl == 1
            ]
            if not correct_texts:
                continue
            correct = correct_texts[0]

            opts = _pad_options(answers, 4)
            try:
                ans_idx = answers.index(correct)
            except ValueError:
                ans_idx = 0
            if ans_idx >= 4:
                ans_idx = 0

            rows.append({
                "article": para,
                "question": q_text,
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx),
            })

    df = pd.DataFrame(rows)
    print(f"[MultiRC] → {len(df):,} rows (super_glue Parquet)")
    return df


# ---------------------------------------------------------------------------
# 7. OpenBookQA  (HuggingFace — 4 options, no passage)
# ---------------------------------------------------------------------------

def fetch_openbookqa() -> pd.DataFrame:
    """Download OpenBookQA from HuggingFace."""
    print("[OBQA] loading allenai/openbookqa …")
    try:
        from datasets import load_dataset
    except ImportError:
        print("  pip install datasets")
        return pd.DataFrame()

    try:
        ds = load_dataset("allenai/openbookqa", "main")
    except Exception as e:
        print(f"  Failed: {e}")
        return pd.DataFrame()

    rows = []
    for split in ds:
        for ex in ds[split]:
            texts = list(ex["choices"]["text"])
            labels = list(ex["choices"]["label"])
            ans_key = str(ex.get("answerKey", "")).strip().upper()
            try:
                ans_idx = labels.index(ans_key)
            except ValueError:
                ans_idx = 0
            opts = _pad_options(texts, 4)
            if ans_idx >= len(opts):
                ans_idx = 0
            rows.append({
                "article": ex["question_stem"],
                "question": "",
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx),
            })
    df = pd.DataFrame(rows)
    print(f"[OBQA]    → {len(df):,} rows (HuggingFace)")
    return df


# ---------------------------------------------------------------------------
# 8. SciQ  (HuggingFace Parquet — 4 options, has support passage)
# ---------------------------------------------------------------------------

def fetch_sciq() -> pd.DataFrame:
    """Download SciQ from HuggingFace Parquet. Has support passage + 4 options."""
    print("[SciQ] loading sciq Parquet …")
    try:
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        print("  pip install huggingface_hub pyarrow")
        return pd.DataFrame()

    import random

    rows = []
    for split in ("train", "validation", "test"):
        try:
            path = hf_hub_download(
                repo_id="sciq",
                filename=f"data/{split}-00000-of-00001.parquet",
                repo_type="dataset",
            )
        except Exception as e:
            print(f"  {split} download failed: {e}")
            continue

        table = pq.read_table(path)
        df_split = table.to_pandas()

        for _, ex in df_split.iterrows():
            opts = [
                ex["correct_answer"],
                ex["distractor1"],
                ex["distractor2"],
                ex["distractor3"],
            ]
            random.shuffle(opts)
            ans_idx = opts.index(ex["correct_answer"])
            rows.append({
                "article": str(ex.get("support", "")),
                "question": ex["question"],
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx),
            })

    df = pd.DataFrame(rows)
    print(f"[SciQ]    → {len(df):,} rows (HuggingFace Parquet)")
    return df


# ---------------------------------------------------------------------------
# 9. MMLU  (HuggingFace Parquet — all subjects, 4 options)
# ---------------------------------------------------------------------------

def fetch_mmlu() -> pd.DataFrame:
    """Download MMLU from HuggingFace Parquet. All 57 subjects, 4 options each."""
    print("[MMLU] loading cais/mmlu Parquet …")
    try:
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        print("  pip install huggingface_hub pyarrow")
        return pd.DataFrame()

    rows = []
    for split in ("auxiliary_train", "dev", "validation", "test"):
        try:
            path = hf_hub_download(
                repo_id="cais/mmlu",
                filename=f"all/{split}-00000-of-00001.parquet",
                repo_type="dataset",
            )
        except Exception as e:
            print(f"  {split} download failed: {e}")
            continue

        table = pq.read_table(path)
        df_split = table.to_pandas()

        for _, ex in df_split.iterrows():
            opts = list(ex["choices"])
            ans_idx = int(ex["answer"])
            if ans_idx < 0 or ans_idx >= len(opts):
                ans_idx = 0
            rows.append({
                "article": ex["question"],
                "question": "",
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx),
            })

    df = pd.DataFrame(rows)
    print(f"[MMLU]    → {len(df):,} rows (HuggingFace Parquet)")
    return df


# ---------------------------------------------------------------------------
# 10. MedQA  (HuggingFace JSONL — USMLE medical MCQs, 4 options)
# ---------------------------------------------------------------------------

def fetch_medqa() -> pd.DataFrame:
    """Download MedQA (USMLE) from HuggingFace JSONL. 4 options, medical context."""
    print("[MedQA] loading GBaker/MedQA-USMLE-4-options-hf …")
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        print("  pip install huggingface_hub")
        return pd.DataFrame()

    import json

    rows = []
    for split in ("train", "dev", "test"):
        try:
            path = hf_hub_download(
                repo_id="GBaker/MedQA-USMLE-4-options-hf",
                filename=f"{split}.json",
                repo_type="dataset",
            )
        except Exception as e:
            print(f"  {split} download failed: {e}")
            continue

        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                ex = json.loads(line)
                opts = [ex["ending0"], ex["ending1"], ex["ending2"], ex["ending3"]]
                ans_idx = int(ex["label"])
                if ans_idx < 0 or ans_idx >= 4:
                    ans_idx = 0
                rows.append({
                    "article": ex["sent1"],
                    "question": ex.get("sent2", ""),
                    "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                    "answer": _answer_letter(ans_idx),
                })

    df = pd.DataFrame(rows)
    print(f"[MedQA]   → {len(df):,} rows (HuggingFace JSONL)")
    return df


# ---------------------------------------------------------------------------
# 11. QASC  (HuggingFace Parquet — 8 options trimmed to 4)
# ---------------------------------------------------------------------------

def fetch_qasc() -> pd.DataFrame:
    """Download QASC from HuggingFace Parquet. 8 options → keep correct + 3."""
    print("[QASC] loading qasc Parquet …")
    try:
        from huggingface_hub import hf_hub_download
        import pyarrow.parquet as pq
    except ImportError:
        print("  pip install huggingface_hub pyarrow")
        return pd.DataFrame()

    import random

    rows = []
    for split in ("train", "validation", "test"):
        try:
            path = hf_hub_download(
                repo_id="qasc",
                filename=f"data/{split}-00000-of-00001.parquet",
                repo_type="dataset",
            )
        except Exception as e:
            print(f"  {split} download failed: {e}")
            continue

        table = pq.read_table(path)
        df_split = table.to_pandas()

        for _, ex in df_split.iterrows():
            texts = list(ex["choices"]["text"])
            labels = list(ex["choices"]["label"])
            ans_key = str(ex["answerKey"]).strip().upper()
            try:
                ans_idx = labels.index(ans_key)
            except ValueError:
                continue

            fact_parts = [str(ex.get("fact1", "")), str(ex.get("fact2", ""))]
            article = " ".join(f for f in fact_parts if f and f != "nan" and f != "None")

            if len(texts) > 4:
                keep = {ans_idx}
                candidates = [i for i in range(len(texts)) if i != ans_idx]
                random.shuffle(candidates)
                keep.update(candidates[:3])
                final_indices = sorted(keep)
                opts = [texts[i] for i in final_indices]
                ans_idx_new = final_indices.index(ans_idx)
            else:
                opts = _pad_options(texts, 4)
                ans_idx_new = ans_idx if ans_idx < 4 else 0

            rows.append({
                "article": article if len(article) > 20 else ex["question"],
                "question": ex["question"] if len(article) > 20 else "",
                "A": opts[0], "B": opts[1], "C": opts[2], "D": opts[3],
                "answer": _answer_letter(ans_idx_new),
            })

    df = pd.DataFrame(rows)
    print(f"[QASC]    → {len(df):,} rows (HuggingFace Parquet)")
    return df


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def fetch_all(
    include_race: bool = True,
    include_dream: bool = True,
    include_commonsenseqa: bool = True,
    include_arc: bool = True,
    include_socialiqa: bool = True,
    include_multirc: bool = True,
    include_openbookqa: bool = True,
    include_sciq: bool = True,
    include_mmlu: bool = True,
    include_medqa: bool = True,
    include_qasc: bool = True,
    output_path: Optional[str] = None,
) -> pd.DataFrame:
    parts = []

    if include_race:
        df = fetch_race()
        if len(df):
            df["source"] = "race"
            parts.append(df)

    if include_dream:
        df = fetch_dream()
        if len(df):
            df["source"] = "dream"
            parts.append(df)

    if include_commonsenseqa:
        df = fetch_commonsenseqa()
        if len(df):
            df["source"] = "commonsenseqa"
            parts.append(df)

    if include_arc:
        df = fetch_arc()
        if len(df):
            df["source"] = "arc"
            parts.append(df)

    if include_socialiqa:
        df = fetch_socialiqa()
        if len(df):
            df["source"] = "socialiqa"
            parts.append(df)

    if include_multirc:
        df = fetch_multirc()
        if len(df):
            df["source"] = "multirc"
            parts.append(df)

    if include_openbookqa:
        df = fetch_openbookqa()
        if len(df):
            df["source"] = "openbookqa"
            parts.append(df)

    if include_sciq:
        df = fetch_sciq()
        if len(df):
            df["source"] = "sciq"
            parts.append(df)

    if include_mmlu:
        df = fetch_mmlu()
        if len(df):
            df["source"] = "mmlu"
            parts.append(df)

    if include_medqa:
        df = fetch_medqa()
        if len(df):
            df["source"] = "medqa"
            parts.append(df)

    if include_qasc:
        df = fetch_qasc()
        if len(df):
            df["source"] = "qasc"
            parts.append(df)

    if not parts:
        print("No datasets loaded. Aborting.")
        sys.exit(1)

    combined = pd.concat(parts, ignore_index=True)
    before = len(combined)
    combined = combined.drop_duplicates(subset=["article", "question"])
    if len(combined) != before:
        print(f"  Dedup removed {before - len(combined):,} rows")

    combined = combined[combined["article"].str.len() > 20]
    combined = combined[(combined["question"].str.len() > 3) | (combined["question"] == "")]
    combined = combined[combined["answer"].isin(["A", "B", "C", "D"])]
    combined = combined.reset_index(drop=True)

    out_path = Path(output_path or (DATA_RAW / "train.csv"))
    combined.to_csv(out_path, index=False)
    print(f"\n  Written {len(combined):,} rows → {out_path}")
    print(f"  Sources: {combined['source'].value_counts().to_dict()}")
    return combined


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Download MCQ datasets → CSV"
    )
    parser.add_argument("--no-race", action="store_true")
    parser.add_argument("--no-dream", action="store_true")
    parser.add_argument("--no-csqa", action="store_true")
    parser.add_argument("--no-arc", action="store_true")
    parser.add_argument("--no-siqa", action="store_true")
    parser.add_argument("--no-multirc", action="store_true")
    parser.add_argument("--no-obqa", action="store_true")
    parser.add_argument("--no-sciq", action="store_true")
    parser.add_argument("--no-mmlu", action="store_true")
    parser.add_argument("--no-medqa", action="store_true")
    parser.add_argument("--no-qasc", action="store_true")
    parser.add_argument("--output", default=None)
    args, _ = parser.parse_known_args()

    fetch_all(
        include_race=not args.no_race,
        include_dream=not args.no_dream,
        include_commonsenseqa=not args.no_csqa,
        include_arc=not args.no_arc,
        include_socialiqa=not args.no_siqa,
        include_multirc=not args.no_multirc,
        include_openbookqa=not args.no_obqa,
        include_sciq=not args.no_sciq,
        include_mmlu=not args.no_mmlu,
        include_medqa=not args.no_medqa,
        include_qasc=not args.no_qasc,
        output_path=args.output,
    )


[RACE] trying HuggingFace ehovy/race (config='all') …


all/validation-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4934 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/87866 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4887 [00:00<?, ? examples/s]

...

[MedQA]   → 12,723 rows (HuggingFace JSONL)
[QASC] loading qasc Parquet …


data/train-00000-of-00001.parquet:   0%|          | 0.00/1.97M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/224k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/158k [00:00<?, ?B/s]

[QASC]    → 9,060 rows (HuggingFace Parquet)
  Dedup removed 10,045 rows

  Written 279,035 rows → /kaggle/working/data/raw/train.csv
  Sources: {'mmlu': 105821, 'race': 97638, 'medqa': 12721, 'sciq': 12252, 'commonsenseqa': 12091, 'dream': 10196, 'qasc': 9002, 'arc': 7743, 'multirc': 6058, 'openbookqa': 5513}


## PREPROCESSING

In [7]:
import math
import os
import pickle
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity as _sk_cos
from tqdm import tqdm

# ─────────────────────────────────────────────
# PATHS (hard-coded project data layout)
# ─────────────────────────────────────────────

try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    BASE_DIR = os.path.normpath(os.path.join(_THIS_DIR, ".."))
except NameError:
    BASE_DIR = os.getcwd()
DATA_ROOT = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_ROOT, "raw")
PROCESSED_DIR = os.path.join(DATA_ROOT, "processed")

os.makedirs(PROCESSED_DIR, exist_ok=True)

DATA_SIZE = None  # set number for debugging

# ─────────────────────────────────────────────
# SAFE CLEANING
# ─────────────────────────────────────────────

def clean_text(text):
    if text is None:
        return ""

    if isinstance(text, float) and np.isnan(text):
        return ""

    text = str(text).lower()
    text = text.replace("nan", "")
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    return text


def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    x = str(x).strip()
    if x.lower() == "nan":
        return ""
    return x


# ─────────────────────────────────────────────
# SENTENCE SPLITTER (FIXED + REQUIRED)
# ─────────────────────────────────────────────

def split_into_sentences(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.replace("\n", " ")
    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [s.strip() for s in sentences if len(s.strip()) > 5]


# ─────────────────────────────────────────────
# TOKENIZER
# ─────────────────────────────────────────────

STOPWORDS = set([
    "a","an","the","is","it","in","on","at","to","for","of",
    "and","or","but","this","that","are","was","were","be"
])

def tokenize(text):
    text = clean_text(text)
    return [w for w in text.split() if w not in STOPWORDS and len(w) > 1]


# ─────────────────────────────────────────────
# LOAD DATA (RACE FORMAT SAFE)
# ─────────────────────────────────────────────

def load_data():
    path = os.path.join(RAW_DIR, "train.csv")

    df = pd.read_csv(path, nrows=DATA_SIZE)

    # SAFE CLEAN ALL COLUMNS
    for col in ["article", "question", "A", "B", "C", "D", "answer"]:
        if col in df.columns:
            df[col] = df[col].apply(safe_str)

    # REMOVE BAD ROWS
    df = df[
        (df["article"].str.len() > 20) &
        (df["question"].str.len() > 3)
    ]

    df = df[df["answer"].isin(["A", "B", "C", "D"])]

    df = df.reset_index(drop=True)

    train, temp = train_test_split(df, test_size=0.2, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)

    return {"train": train, "val": val, "test": test}


# ─────────────────────────────────────────────
# TF-IDF (MODEL A CORE FEATURE)
# ─────────────────────────────────────────────

def build_tfidf_vectorizer(texts):
    vec = TfidfVectorizer(
        max_features=10000,
        ngram_range=(1,2),
        stop_words="english"
    )
    vec.fit(texts)
    return vec


def tfidf_cosine(a, b, vec):
    a = clean_text(a)
    b = clean_text(b)

    if not a or not b:
        return 0.0

    v = vec.transform([a, b])
    return float(cosine_similarity(v[0], v[1])[0][0])


def tfidf_cosine_similarity(text_a, text_b, vectorizer):
    """Alias expected by model_a_train (same as tfidf_cosine)."""
    return tfidf_cosine(text_a, text_b, vectorizer)


def cosine_similarity_feature(text_a, text_b):
    a, b = Counter(tokenize(text_a)), Counter(tokenize(text_b))
    if not a or not b:
        return 0.0
    dot = sum(a[k] * b[k] for k in a if k in b)
    denom = math.sqrt(sum(v**2 for v in a.values())) * math.sqrt(
        sum(v**2 for v in b.values())
    ) + 1e-9
    return dot / denom


def build_one_sample(article, question, option):
    return clean_text(f"{article} {question} {option}")


def encode_texts(texts, vectorizer):
    return vectorizer.transform(texts)


# ─────────────────────────────────────────────
# MODEL A DATASET BUILDER (FIXED)
# ─────────────────────────────────────────────

def build_model_a_dataset(df, tfidf_vec=None):
    texts = []
    labels = []
    hc_feats = []

    idf_dict = None
    if tfidf_vec is not None:
        idf_dict = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_vec.idf_))
        _vocab = set(tfidf_vec.get_feature_names_out())

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  dataset rows", unit="row"):
        gold = str(row["answer"]).strip().upper()
        article = clean_text(row["article"])
        question = clean_text(row["question"])
        art_tokens = set(article.split())
        q_tokens = set(question.split())
        art_bigrams = set(
            " ".join(article.split()[i:i+2])
            for i in range(len(article.split()) - 1)
        )

        for opt in ["A", "B", "C", "D"]:
            opt_text = clean_text(row[opt])
            if opt_text == "":
                continue

            combined = article + " " + question + " " + opt_text
            texts.append(combined)
            labels.append(1 if opt == gold else 0)

            if tfidf_vec is not None and len(opt_text) > 0:
                art_vec = tfidf_vec.transform([article])
                q_vec = tfidf_vec.transform([question])
                opt_vec = tfidf_vec.transform([opt_text])
                sim_ao = float(_sk_cos(art_vec, opt_vec)[0, 0])
                sim_qo = float(_sk_cos(q_vec, opt_vec)[0, 0])
                opt_tfidf_max = float(opt_vec.max())
            else:
                sim_ao = sim_qo = 0.0
                opt_tfidf_max = 0.0

            opt_tokens = set(opt_text.split())
            overlap_ao = len(opt_tokens & art_tokens) / (len(opt_tokens) + 1e-9)
            overlap_qo = len(opt_tokens & q_tokens) / (len(opt_tokens) + 1e-9)
            opt_len_ratio = min(len(opt_text) / 200.0, 1.0)
            overlap_oa = len(opt_tokens & art_tokens) / (len(art_tokens) + 1e-9)
            overlap_oq = len(opt_tokens & q_tokens) / (len(q_tokens) + 1e-9)

            opt_idf_mean = 0.0
            if idf_dict and opt_tokens:
                idfs = [idf_dict.get(w, 0.0) for w in opt_tokens if w in _vocab]
                opt_idf_mean = sum(idfs) / len(idfs) if idfs else 0.0

            has_digit = 1.0 if re.search(r"\d", opt_text) else 0.0

            opt_bigrams = set(
                " ".join(opt_text.split()[i:i+2])
                for i in range(len(opt_text.split()) - 1)
            )
            shared_bigrams = len(art_bigrams & opt_bigrams) / (len(opt_bigrams) + 1e-9) if opt_bigrams else 0.0

            starts_cap = 1.0 if opt_text and opt_text[0].isupper() else 0.0

            hc_feats.append([
                sim_ao, sim_qo,
                overlap_ao, overlap_qo,
                opt_len_ratio,
                overlap_oa, overlap_oq,
                opt_idf_mean,
                has_digit,
                shared_bigrams,
                opt_tfidf_max,
                starts_cap,
            ])

    return texts, labels, np.array(hc_feats, dtype=np.float32)


# ─────────────────────────────────────────────
# OHE MATRICES + VECTORIZERS (Model A / B on disk)
# ─────────────────────────────────────────────

def build_training_artifacts(processed_dir, max_ohe_features=10000):
    """Sparse OHE + handcrafted features, labels, vectorizers."""
    for split in ("train", "val", "test"):
        path = os.path.join(processed_dir, f"{split}_clean.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Missing {path}; run run_preprocessing() first."
            )

    train_df = pd.read_csv(os.path.join(processed_dir, "train_clean.csv"))
    val_df = pd.read_csv(os.path.join(processed_dir, "val_clean.csv"))
    test_df = pd.read_csv(os.path.join(processed_dir, "test_clean.csv"))

    tfidf_pkl = os.path.join(processed_dir, "tfidf.pkl")
    tfidf_vec = None
    if os.path.exists(tfidf_pkl):
        with open(tfidf_pkl, "rb") as f:
            tfidf_vec = pickle.load(f)

    tr_texts, tr_y, hc_tr = build_model_a_dataset(train_df, tfidf_vec)
    va_texts, va_y, hc_va = build_model_a_dataset(val_df, tfidf_vec)
    te_texts, te_y, hc_te = build_model_a_dataset(test_df, tfidf_vec)

    vec = CountVectorizer(max_features=max_ohe_features, ngram_range=(1, 2))
    vec.fit(tr_texts)

    X_tr = vec.transform(tr_texts)
    X_va = vec.transform(va_texts)
    X_te = vec.transform(te_texts)

    y_tr = np.asarray(tr_y, dtype=np.int64)
    y_va = np.asarray(va_y, dtype=np.int64)
    y_te = np.asarray(te_y, dtype=np.int64)

    save_npz(os.path.join(processed_dir, "X_train_ohe.npz"), X_tr)
    save_npz(os.path.join(processed_dir, "X_val_ohe.npz"), X_va)
    save_npz(os.path.join(processed_dir, "X_test_ohe.npz"), X_te)
    np.save(os.path.join(processed_dir, "y_train.npy"), y_tr)
    np.save(os.path.join(processed_dir, "y_val.npy"), y_va)
    np.save(os.path.join(processed_dir, "y_test.npy"), y_te)
    np.save(os.path.join(processed_dir, "hc_train.npy"), hc_tr)
    np.save(os.path.join(processed_dir, "hc_val.npy"), hc_va)
    np.save(os.path.join(processed_dir, "hc_test.npy"), hc_te)

    with open(os.path.join(processed_dir, "ohe_vectorizer.pkl"), "wb") as f:
        pickle.dump(vec, f)

    # Copy or validate tfidf_vectorizer.pkl for model_b
    tfidf_out = os.path.join(processed_dir, "tfidf_vectorizer.pkl")
    if tfidf_vec is not None and not os.path.exists(tfidf_out):
        with open(tfidf_out, "wb") as f:
            pickle.dump(tfidf_vec, f)
    elif not os.path.exists(tfidf_out):
        raise FileNotFoundError(
            f"Expected {tfidf_pkl} or existing {tfidf_out} for model_b_train."
        )


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

def run_preprocessing(max_ohe_features=10000):

    print("Loading data...")
    splits = load_data()

    train = splits["train"]
    val = splits["val"]
    test = splits["test"]

    print("Building TF-IDF...")
    tfidf = build_tfidf_vectorizer(train["article"].astype(str))

    print("Saving cleaned datasets...")

    train.to_csv(os.path.join(PROCESSED_DIR, "train_clean.csv"), index=False)
    val.to_csv(os.path.join(PROCESSED_DIR, "val_clean.csv"), index=False)
    test.to_csv(os.path.join(PROCESSED_DIR, "test_clean.csv"), index=False)

    with open(os.path.join(PROCESSED_DIR, "tfidf.pkl"), "wb") as f:
        pickle.dump(tfidf, f)

    print("Building OHE matrices and tfidf_vectorizer.pkl …")
    build_training_artifacts(PROCESSED_DIR, max_ohe_features=max_ohe_features)

    print("\nDONE — PREPROCESSING SUCCESSFUL (MODEL A READY)")


# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    run_preprocessing()

Loading data...


/tmp/ipykernel_57/4112848730.py:98: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, nrows=DATA_SIZE)


Building TF-IDF...


KeyboardInterrupt: 

## NN MODELS.PY


In [1]:
"""
nn_models.py — PyTorch neural network models & training utilities.
Replaces scikit-learn classifiers (Logistic Regression, SVM, meta-learner)
with feed-forward networks for the RACE RC project.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
from scipy.sparse import issparse
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import sys as _sys
print(f"[nn_models] device={DEVICE}", file=_sys.stderr)
if torch.cuda.is_available():
    print(f"[nn_models]   GPU: {torch.cuda.get_device_name(0)}", file=_sys.stderr)
    print(f"[nn_models]   CUDA: {torch.version.cuda}", file=_sys.stderr)


# ---------------------------------------------------------------------------
# Model unwrapping utility
# ---------------------------------------------------------------------------

def _unwrap_model(model: nn.Module) -> nn.Module:
    """
    Peel off DataParallel, DistributedDataParallel, and torch.compile
    (OptimizedModule) wrappers to get the original nn.Module back.

    Call this whenever you need to access custom attributes (.tokenizer,
    .max_input_length, etc.) or save a portable state_dict.

    Safe to call on an already-unwrapped model.
    """
    # torch.compile wraps the model in torch._dynamo.eval_frame.OptimizedModule
    # which exposes the original via ._orig_mod
    while hasattr(model, "_orig_mod"):
        model = model._orig_mod
    # DataParallel / DistributedDataParallel expose the original via .module
    while isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)):
        model = model.module
    # A compile wrapper may sit on top of a DataParallel, so alternate until stable
    # (the while loops above handle arbitrary nesting already)
    return model


# ---------------------------------------------------------------------------
# Dataset wrapper
# ---------------------------------------------------------------------------

class SparseDataset(Dataset):
    """Wraps a sparse CSR matrix (or dense array) with binary labels."""

    def __init__(self, X, y):
        self.X = X
        self.y = np.asarray(y, dtype=np.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        row = self.X[idx]
        if issparse(row):
            x = row.toarray().flatten().astype(np.float32)
        else:
            x = np.asarray(row).flatten().astype(np.float32)
        return torch.from_numpy(x), torch.tensor(self.y[idx], dtype=torch.float32)


# ---------------------------------------------------------------------------
# Base mixin providing sklearn-compatible predict / predict_proba
# ---------------------------------------------------------------------------

class _BaseNN(nn.Module):
    """Adds .predict() and .predict_proba() so callers need minimal changes."""

    def predict_proba(self, X, batch_size=1024):
        """Return (N, 2) — [P(0), P(1)] matching sklearn's interface."""
        self.eval()
        ds = SparseDataset(X, np.zeros(X.shape[0], dtype=np.float32))
        ld = DataLoader(ds, batch_size=batch_size, shuffle=False)
        all_p1 = []
        with torch.no_grad():
            for bx, _ in ld:
                logits = self.forward(bx.to(DEVICE))
                all_p1.append(torch.sigmoid(logits).cpu().numpy())
        p1 = np.concatenate(all_p1)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X, batch_size=1024):
        """Return binary class predictions (0 / 1)."""
        return np.argmax(self.predict_proba(X, batch_size), axis=1)


# ---------------------------------------------------------------------------
# Model A  —  Answer Verifier (binary classifier on OHE features)
# ---------------------------------------------------------------------------

class AnswerVerifier(_BaseNN):
    """
    Feed-forward network for binary answer verification.

    Input   : OHE / bag-of-words vector (default 10 000 dims)
    Arch    : Linear → BN → ReLU → Dropout → … → Linear(→1)
    Output  : logit (fed into sigmoid in predict_proba)
    """

    def __init__(self, input_dim: int, hidden_dims=None, dropout: float = 0.3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [512, 128]
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ---------------------------------------------------------------------------
# Model B  —  Distractor Scorer & Hint Scorer (small NNs on handcrafted feats)
# ---------------------------------------------------------------------------

class DistractorScorer(_BaseNN):
    """Small network for distractor scoring (6 handcrafted features)."""

    def __init__(self, input_dim: int = 6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class HintScorer(_BaseNN):
    """Small network for hint scoring (4 handcrafted features)."""

    def __init__(self, input_dim: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ---------------------------------------------------------------------------
# Training helpers
# ---------------------------------------------------------------------------

def _train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    total = 0.0
    use_amp = device.type == "cuda" and scaler is not None
    for bx, by in loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        optimizer.zero_grad()
        if use_amp:
            with autocast("cuda"):
                loss = criterion(model(bx), by)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
        total += loss.item() * bx.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def _eval_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for bx, by in loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        logits = model(bx)
        total_loss += criterion(logits, by).item() * bx.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        all_preds.append(preds.cpu().numpy())
        all_labels.append(by.cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader.dataset), float((preds == labels).mean()), preds


def _checkpoint_payload(
    model: nn.Module,
    epoch: int,
    optimizer: torch.optim.Optimizer,
    scheduler,
    best_loss: float,
    best_state,
    stall: int,
    extra_meta: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    return {
        "epoch": epoch,
        "model_state_dict": _unwrap_model(model).state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "best_loss": best_loss,
        "best_state_dict": best_state,
        "stall": stall,
        "meta": extra_meta or {},
    }


def _save_epoch_checkpoint(
    checkpoint_dir: str | Path,
    model: nn.Module,
    epoch: int,
    optimizer: torch.optim.Optimizer,
    scheduler,
    best_loss: float,
    best_state,
    stall: int,
    extra_meta: Optional[Dict[str, Any]] = None,
) -> Path:
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    payload = _checkpoint_payload(
        model=model,
        epoch=epoch,
        optimizer=optimizer,
        scheduler=scheduler,
        best_loss=best_loss,
        best_state=best_state,
        stall=stall,
        extra_meta=extra_meta,
    )
    epoch_path = checkpoint_dir / f"epoch_{epoch:04d}.ckpt"
    latest_path = checkpoint_dir / "latest.ckpt"
    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    print(f"  checkpoint saved → {epoch_path}")
    return latest_path


def _load_training_checkpoint(
    checkpoint_path: str | Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return 1, float("inf"), None, 0

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    _unwrap_model(model).load_state_dict(ckpt["model_state_dict"])
    if ckpt.get("optimizer_state_dict"):
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler_state = ckpt.get("scheduler_state_dict")
    if scheduler is not None and scheduler_state:
        scheduler.load_state_dict(scheduler_state)

    start_epoch = int(ckpt.get("epoch", 0)) + 1
    best_loss = float(ckpt.get("best_loss", float("inf")))
    best_state = ckpt.get("best_state_dict")
    stall = int(ckpt.get("stall", 0))
    return start_epoch, best_loss, best_state, stall


def train_nn(
    model: nn.Module,
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    epochs: int = 30,
    batch_size: int = 256,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    pos_weight: Optional[float] = None,
    patience: int = 5,
    checkpoint_dir: str | Path | None = None,
    verbose: bool = True,
) -> nn.Module:
    """
    Full training loop with early stopping.

    Parameters
    ----------
    model        : a PyTorch module (must have forward returning logits)
    X_train, y_train : training data (sparse or dense)
    X_val, y_val : optional validation data
    epochs       : maximum number of epochs
    batch_size   : mini-batch size
    lr, weight_decay : AdamW hyper-parameters
    pos_weight   : weight for the positive class in BCEWithLogitsLoss
                   (set to neg_count / pos_count for imbalanced data)
    patience     : early stopping patience (validation loss)
    """
    device = DEVICE
    nw = 2 if device.type == "cuda" else 0
    train_ds = SparseDataset(X_train, y_train)
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=(nw > 0), num_workers=nw)

    val_ld = None
    if X_val is not None and y_val is not None:
        val_ds = SparseDataset(X_val, y_val)
        val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=(nw > 0), num_workers=nw)

    if pos_weight is not None:
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], device=device)
        )
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    scaler = GradScaler("cuda") if device.type == "cuda" else None
    model = model.to(device)
    best_loss = float("inf")
    best_state = None
    stall = 0

    pbar = tqdm(range(1, epochs + 1), desc="  epochs", unit="ep", disable=not verbose)
    for epoch in pbar:
        tr_loss = _train_epoch(model, train_ld, criterion, optimizer, device, scaler)
        scheduler.step()

        if val_ld is not None:
            val_loss, val_acc, _ = _eval_model(model, val_ld, criterion, device)
            pbar.set_postfix(tr_loss=f"{tr_loss:.4f}", val_loss=f"{val_loss:.4f}", val_acc=f"{val_acc:.4f}")
            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                stall = 0
            else:
                stall += 1
                if stall >= patience:
                    if verbose:
                        print(f"  early stopping at epoch {epoch}")
                    break
        else:
            pbar.set_postfix(tr_loss=f"{tr_loss:.4f}")

        if checkpoint_dir is not None:
            _save_epoch_checkpoint(
                checkpoint_dir=checkpoint_dir,
                model=model,
                epoch=epoch,
                optimizer=optimizer,
                scheduler=scheduler,
                best_loss=best_loss,
                best_state=best_state,
                stall=stall,
                extra_meta={"trainer": "nn", "epoch": epoch},
            )

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


# ---------------------------------------------------------------------------
# Checkpoint save / load
# ---------------------------------------------------------------------------

def save_checkpoint(model: nn.Module, path: str, extra_meta: Optional[Dict[str, Any]] = None):
    """Save model state_dict + metadata."""
    base = _unwrap_model(model)
    meta: Dict[str, Any] = {
        "input_dim": (
            base.net[0].in_features
            if hasattr(base, "net") and hasattr(base.net[0], "in_features")
            else None
        )
    }
    if extra_meta:
        meta.update(extra_meta)
    torch.save({"model_state_dict": base.state_dict(), "meta": meta}, path)


def load_checkpoint(model_class, path: str, **model_kwargs):
    """Load a saved checkpoint into a fresh model instance."""
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        ckpt = torch.load(path, map_location="cpu")
    meta = ckpt.get("meta", {})
    input_dim = meta.get("input_dim") or model_kwargs.pop("input_dim", None)
    if input_dim is None and "input_dim" not in model_kwargs:
        raise ValueError(
            "input_dim must be provided via model_kwargs or checkpoint meta"
        )
    model = model_class(input_dim=input_dim, **model_kwargs)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


# ===================================================================
# Transformer-based models  (BERT / T5 / BART  for MCQ pipeline)
# ===================================================================


class TransformerAnswerVerifier(nn.Module):
    """
    BERT/RoBERTa-based answer verifier.
    Encodes (article, question, option) and scores P(correct).
    Supports LoRA via model_name="<path>/lora-adapter" or HF hub.

    NOTE: DataParallel wrapping is handled externally by transformer_train.py.
          Do NOT wrap self.encoder with DataParallel here — the outer wrapper
          in _wrap_dataparallel() already covers it.
    """

    def __init__(
        self,
        model_name: str = "bert-base-uncased",
        num_labels: int = 2,
        max_length: int = 384,
        use_lora: bool = False,
        lora_r: int = 8,
    ):
        super().__init__()
        from transformers import AutoConfig, AutoModelForSequenceClassification

        self.max_length = max_length
        self.config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
        self.encoder = AutoModelForSequenceClassification.from_pretrained(
            model_name, config=self.config
        )

        if use_lora:
            try:
                from peft import LoraConfig, get_peft_model

                lora_cfg = LoraConfig(
                    r=lora_r,
                    lora_alpha=lora_r * 2,
                    target_modules=["query", "value"],
                    lora_dropout=0.1,
                    bias="none",
                )
                self.encoder = get_peft_model(self.encoder, lora_cfg)
                self.encoder.print_trainable_parameters()
            except ImportError:
                print("  WARNING: peft not installed. Training full model.")

        # ── Removed internal DataParallel wrap ──────────────────────────────
        # transformer_train.py calls _wrap_dataparallel() on the whole model
        # object after construction, which is the correct place to do it.
        # Wrapping self.encoder here AND then wrapping the whole model in
        # transformer_train.py produces nested DataParallel which breaks
        # attribute access and doubles GPU memory overhead.

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        return out


class TextDataset(Dataset):
    """Simple text-pair dataset for transformer training.

    Tokenises in chunks (default 10 000) with a progress bar so that
    large corpora don't appear to "hang" and don't OOM.
    """

    _CHUNK = 10_000  # texts per tokenizer call

    def __init__(self, texts, labels, tokenizer, max_length=384):
        self.labels = torch.tensor(labels, dtype=torch.long)

        all_ids, all_mask = [], []
        n = len(texts)
        chunks = range(0, n, self._CHUNK)
        for start in tqdm(chunks, desc="  tokenizing", unit="chunk"):
            end = min(start + self._CHUNK, n)
            enc = tokenizer(
                texts[start:end],
                truncation=True,
                padding="max_length",
                max_length=max_length,
                return_tensors="pt",
            )
            all_ids.append(enc["input_ids"])
            all_mask.append(enc["attention_mask"])

        self.input_ids = torch.cat(all_ids, dim=0)
        self.attention_mask = torch.cat(all_mask, dim=0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def train_transformer(
    model: nn.Module,
    tokenizer,
    train_texts,
    train_labels,
    val_texts=None,
    val_labels=None,
    epochs: int = 5,
    batch_size: int = 48,
    lr: float = 2e-5,
    max_length: int = 384,
    patience: int = 2,
    checkpoint_dir: str | Path = "./checkpoints",
    resume_from_checkpoint: str | Path | None = None,
    verbose: bool = True,
):
    """Training loop for TransformerAnswerVerifier with early stopping."""
    device = DEVICE
    model = model.to(device)
    nw = min(8, os.cpu_count() or 1)

    train_ds = TextDataset(train_texts, train_labels, tokenizer, max_length)
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=nw)

    val_ld = None
    if val_texts is not None and val_labels is not None:
        val_ds = TextDataset(val_texts, val_labels, tokenizer, max_length)
        val_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=nw)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler("cuda") if device.type == "cuda" else None
    use_amp = scaler is not None

    start_epoch = 1
    best_loss = float("inf")
    best_state = None
    stall = 0
    n_batches = len(train_ld)

    if resume_from_checkpoint is not None:
        start_epoch, best_loss, best_state, stall = _load_training_checkpoint(
            resume_from_checkpoint,
            model,
            optimizer,
            scheduler,
        )
        if verbose:
            tqdm.write(f"  resumed from epoch {start_epoch - 1} using {resume_from_checkpoint}")

    if start_epoch > epochs:
        if verbose:
            tqdm.write(f"  checkpoint is already at epoch {start_epoch - 1}; nothing to do")
        if best_state is not None:
            _unwrap_model(model).load_state_dict(best_state)
        return model

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_ld, total=n_batches, desc=f"  epoch {epoch}/{epochs}", unit="batch", leave=False)
        for batch in pbar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            optimizer.zero_grad()
            if use_amp:
                with autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                scaler.scale(out.loss.mean()).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                out.loss.mean().backward()
                optimizer.step()
            tr_loss += out.loss.mean().item() * input_ids.size(0)
            pbar.set_postfix(loss=f"{out.loss.mean().item():.4f}")
        scheduler.step()
        tr_loss /= len(train_ld.dataset)

        if val_ld is not None:
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            with torch.no_grad():
                for batch in val_ld:
                    input_ids = batch["input_ids"].to(device, non_blocking=True)
                    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                    labels = batch["labels"].to(device, non_blocking=True)
                    if use_amp:
                        with autocast("cuda"):
                            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    else:
                        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    val_loss += out.loss.mean().item() * input_ids.size(0)
                    preds = out.logits.argmax(-1)
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)
            val_loss /= len(val_ld.dataset)
            val_acc = correct / total if total > 0 else 0.0

            if verbose:
                tqdm.write(f"  epoch {epoch:2d}/{epochs}  tr_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                stall = 0
            else:
                stall += 1
        else:
            if verbose:
                tqdm.write(f"  epoch {epoch:2d}/{epochs}  tr_loss={tr_loss:.4f}")

        if checkpoint_dir is not None:
            _save_epoch_checkpoint(
                checkpoint_dir=checkpoint_dir,
                model=model,
                epoch=epoch,
                optimizer=optimizer,
                scheduler=scheduler,
                best_loss=best_loss,
                best_state=best_state,
                stall=stall,
                extra_meta={"trainer": "transformer", "epoch": epoch},
            )

        if val_ld is not None and stall >= patience:
            if verbose:
                tqdm.write(f"  early stopping at epoch {epoch}")
            break

    if best_state is not None:
        _unwrap_model(model).load_state_dict(best_state)
    return model


# ===================================================================
# Seq2Seq models for MCQ generation (T5 / BART / FLAN-T5)
# ===================================================================


class QuestionGenerator(nn.Module):
    """
    T5/FLAN-T5 based question generator.
    Input  : "context: ... answer: ..."
    Output : generated question text
    Supports LoRA.

    NOTE: DataParallel wrapping is handled externally by transformer_train.py.
          Do NOT wrap self.model with DataParallel here.
    """

    def __init__(
        self,
        model_name: str = "google/flan-t5-base",
        max_input_length: int = 384,
        max_output_length: int = 64,
        use_lora: bool = False,
        lora_r: int = 8,
    ):
        super().__init__()
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length

        if use_lora:
            try:
                from peft import LoraConfig, get_peft_model

                lora_cfg = LoraConfig(
                    r=lora_r,
                    lora_alpha=lora_r * 2,
                    target_modules=["q", "v"],
                    lora_dropout=0.1,
                    bias="none",
                )
                self.model = get_peft_model(self.model, lora_cfg)
                self.model.print_trainable_parameters()
            except ImportError:
                print("  WARNING: peft not installed. Training full model.")

        # ── Removed internal DataParallel wrap ──────────────────────────────
        # See TransformerAnswerVerifier note above.

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

    def generate_question(self, context: str, answer: str) -> str:
        # Always unwrap to get the raw HF model for .generate()
        base = _unwrap_model(self)
        prompt = f"context: {context} answer: {answer}"
        inputs = base.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=base.max_input_length,
        ).to(next(base.model.parameters()).device)
        outputs = base.model.generate(
            **inputs,
            max_new_tokens=base.max_output_length,
            num_beams=4,
            temperature=0.7,
        )
        return base.tokenizer.decode(outputs[0], skip_special_tokens=True)


class DistractorGenerator(nn.Module):
    """
    T5/BART based distractor generator.
    Input  : "question: ... correct: ..."
    Output : distractor texts separated by |
    Supports LoRA.

    NOTE: DataParallel wrapping is handled externally by transformer_train.py.
          Do NOT wrap self.model with DataParallel here.
    """

    def __init__(
        self,
        model_name: str = "facebook/bart-base",
        max_input_length: int = 256,
        max_output_length: int = 96,
        use_lora: bool = False,
        lora_r: int = 8,
    ):
        super().__init__()
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length

        if use_lora:
            try:
                from peft import LoraConfig, get_peft_model

                lora_cfg = LoraConfig(
                    r=lora_r,
                    lora_alpha=lora_r * 2,
                    target_modules=["q_proj", "v_proj"],
                    lora_dropout=0.1,
                    bias="none",
                )
                self.model = get_peft_model(self.model, lora_cfg)
                self.model.print_trainable_parameters()
            except ImportError:
                print("  WARNING: peft not installed. Training full model.")

        # ── Removed internal DataParallel wrap ──────────────────────────────
        # See TransformerAnswerVerifier note above.

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

    def generate_distractors(self, question: str, correct_answer: str, n: int = 3) -> list:
        # Always unwrap to get the raw HF model for .generate()
        base = _unwrap_model(self)
        prompt = f"question: {question} correct: {correct_answer}"
        inputs = base.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=base.max_input_length,
        ).to(next(base.model.parameters()).device)
        outputs = base.model.generate(
            **inputs,
            max_new_tokens=base.max_output_length,
            num_beams=4,
            temperature=0.8,
        )
        text = base.tokenizer.decode(outputs[0], skip_special_tokens=True)
        distractors = [d.strip() for d in text.split("|") if d.strip()]
        while len(distractors) < n:
            distractors.append(f"option {len(distractors) + 1}")
        return distractors[:n]


def train_seq2seq(
    model: nn.Module,
    train_inputs: list,
    train_targets: list,
    val_inputs: list = None,
    val_targets: list = None,
    epochs: int = 10,
    batch_size: int = 24,
    lr: float = 3e-5,
    patience: int = 2,
    checkpoint_dir: str | Path = "./checkpoints",
    resume_from_checkpoint: str | Path | None = None,
    verbose: bool = True,
):
    """
    Training loop for seq2seq models (QuestionGenerator / DistractorGenerator).
    Each input is raw text; each target is the expected output text.

    `model` may be wrapped with DataParallel and/or torch.compile by the caller
    (transformer_train.py).  We unwrap here to safely access .tokenizer,
    .max_input_length, and .max_output_length before the training loop begins.
    """
    device = DEVICE
    model = model.to(device)

    # ── Unwrap to access custom attributes safely ───────────────────────────
    # After _wrap_dataparallel() and _try_compile() in transformer_train.py,
    # `model` may be:
    #   OptimizedModule(_orig_mod=DataParallel(module=QuestionGenerator(...)))
    # _unwrap_model() peels all layers to reach the original QuestionGenerator.
    base_model = _unwrap_model(model)
    tokenizer = base_model.tokenizer
    max_in = base_model.max_input_length
    max_out = base_model.max_output_length

    nw = min(8, os.cpu_count() or 2) if device.type == "cuda" else 0

    def _encode(texts, max_len):
        return tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_len, return_tensors="pt",
        )

    train_enc = _encode(train_inputs, max_in)
    train_labels = _encode(train_targets, max_out).input_ids
    train_labels[train_labels == tokenizer.pad_token_id] = -100

    val_data = None
    if val_inputs and val_targets:
        val_enc = _encode(val_inputs, max_in)
        val_labels = _encode(val_targets, max_out).input_ids
        val_labels[val_labels == tokenizer.pad_token_id] = -100
        val_data = (val_enc, val_labels)

    class _Seq2SeqDataset(Dataset):
        def __init__(self, enc, labels):
            self.enc = enc
            self.labels = labels

        def __len__(self):
            return self.enc.input_ids.size(0)

        def __getitem__(self, idx):
            return {
                "input_ids": self.enc.input_ids[idx],
                "attention_mask": self.enc.attention_mask[idx],
                "labels": self.labels[idx],
            }

    train_ds = _Seq2SeqDataset(train_enc, train_labels)
    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=(nw > 0), num_workers=nw)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler("cuda") if device.type == "cuda" else None
    use_amp = scaler is not None

    start_epoch = 1
    best_loss = float("inf")
    best_state = None
    stall = 0
    n_batches = len(train_ld)

    if resume_from_checkpoint is not None:
        start_epoch, best_loss, best_state, stall = _load_training_checkpoint(
            resume_from_checkpoint,
            model,
            optimizer,
            scheduler,
        )
        if verbose:
            tqdm.write(f"  resumed from epoch {start_epoch - 1} using {resume_from_checkpoint}")

    if start_epoch > epochs:
        if verbose:
            tqdm.write(f"  checkpoint is already at epoch {start_epoch - 1}; nothing to do")
        if best_state is not None:
            _unwrap_model(model).load_state_dict(best_state)
        return model

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        tr_loss = 0.0
        pbar = tqdm(train_ld, total=n_batches, desc=f"  epoch {epoch}/{epochs}", unit="batch", leave=False)
        for batch in pbar:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            optimizer.zero_grad()
            if use_amp:
                with autocast("cuda"):
                    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                scaler.scale(out.loss.mean()).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                out.loss.mean().backward()
                optimizer.step()
            tr_loss += out.loss.mean().item() * input_ids.size(0)
            pbar.set_postfix(loss=f"{out.loss.mean().item():.4f}")
        tr_loss /= len(train_ld.dataset)
        scheduler.step()

        if val_data is not None:
            model.eval()
            val_enc, val_lbl = val_data
            val_loss = 0.0
            with torch.no_grad():
                if use_amp:
                    with autocast("cuda"):
                        out = model(
                            input_ids=val_enc.input_ids.to(device, non_blocking=True),
                            attention_mask=val_enc.attention_mask.to(device, non_blocking=True),
                            labels=val_lbl.to(device, non_blocking=True),
                        )
                else:
                    out = model(
                        input_ids=val_enc.input_ids.to(device, non_blocking=True),
                        attention_mask=val_enc.attention_mask.to(device, non_blocking=True),
                        labels=val_lbl.to(device, non_blocking=True),
                    )
                val_loss = out.loss.mean().item()
            if verbose:
                tqdm.write(f"  epoch {epoch:2d}/{epochs}  tr_loss={tr_loss:.4f}  val_loss={val_loss:.4f}")
            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                stall = 0
            else:
                stall += 1
        else:
            if verbose:
                tqdm.write(f"  epoch {epoch:2d}/{epochs}  tr_loss={tr_loss:.4f}")

        if checkpoint_dir is not None:
            _save_epoch_checkpoint(
                checkpoint_dir=checkpoint_dir,
                model=model,
                epoch=epoch,
                optimizer=optimizer,
                scheduler=scheduler,
                best_loss=best_loss,
                best_state=best_state,
                stall=stall,
                extra_meta={"trainer": "seq2seq", "epoch": epoch},
            )

        if val_data is not None and stall >= patience:
            if verbose:
                tqdm.write(f"  early stopping at epoch {epoch}")
            break

    if best_state is not None:
        _unwrap_model(model).load_state_dict(best_state)
    return model

[nn_models] device=cuda
[nn_models]   GPU: Tesla T4
[nn_models]   CUDA: 12.8


## MODEL A TRAINING


In [ ]:
"""
model_a_train.py
================
Answer Verification + MCQ Generation  (Model A)

Models:
  AnswerVerifier (feed-forward NN on OHE features)
  TransformerAnswerVerifier (BERT/RoBERTa + LoRA)  — optional

MCQ generation pipeline:
  Phase 1 — candidate sentence extraction (keyword-overlap scoring)
  Phase 2 — Wh-word template instantiation
  Phase 3 — ML / heuristic question ranking
  Phase 4 — distractor assembly from article sentences

Evaluation:
  Binary classification: Accuracy, Precision, Recall, Macro-F1
  4-way MCQ accuracy
  Cosine-sim accuracy (TF-IDF)
  Text generation: BLEU, ROUGE-1/2/L, METEOR
"""

from __future__ import annotations

import math
import os
import pickle
import random
import re
import sys
import warnings
from collections import Counter
from itertools import chain
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
from scipy.sparse import load_npz, hstack as sparse_hstack
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity as _sk_cos
from tqdm import tqdm

# from src.nn_models import (
#     AnswerVerifier,
#     DistractorScorer,
#     HintScorer,
#     DEVICE,
#     load_checkpoint,
#     save_checkpoint,
#     train_nn,
# )
# from src.preprocessing import clean_text

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Project imports
# ---------------------------------------------------------------------------

# Local development path (commented)
# _THIS_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else "/kaggle/working/data"
# ARTIFACT_DIR = os.path.join(_THIS_DIR, "..", "data", "processed")

# Local paths
try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    BASE_PROJ  = os.path.normpath(os.path.join(_THIS_DIR, ".."))
except NameError:
    BASE_PROJ = os.getcwd()
ARTIFACT_DIR = os.path.join(BASE_PROJ, "data", "processed")
PROCESSED_DIR = ARTIFACT_DIR

sys.path.insert(0, _THIS_DIR)
# try:
#     from preprocessing import (
#         ARTIFACT_DIR,
#         normalize,
#         word_tokens,
#         sentence_fragments,
#         dot_cosine,
#         vec_cosine,
#         build_model_a_dataset,
#         apply_vectorizer,
#         make_sample_string,
#     )
#     PROCESSED_DIR = ARTIFACT_DIR
# except ImportError:
#     # Minimal stubs so the module loads without preprocessing.py
#     ARTIFACT_DIR  = os.path.join(_THIS_DIR, "processed")
#     PROCESSED_DIR = ARTIFACT_DIR
_SW = frozenset({
    "a", "an", "the", "is", "it", "in", "of", "to", "and", "or", "for",
    "on", "with", "as", "at", "by", "be", "was", "are", "were", "this",
    "that", "from", "but", "not", "have", "has", "had", "he", "she",
    "they", "we", "you", "i", "do", "did", "will", "its", "their",
})
def normalize(raw) -> str:
    s = str(raw).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def word_tokens(text: str) -> List[str]:
    return [t for t in normalize(text).split() if t not in _SW and len(t) > 1]
def sentence_fragments(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", str(text).strip())
    return [p.strip() for p in parts if len(p.strip()) > 10]
def dot_cosine(a: str, b: str) -> float:
    ba, bb = Counter(word_tokens(a)), Counter(word_tokens(b))
    if not ba or not bb:
        return 0.0
    dot  = sum(ba[k] * bb[k] for k in ba if k in bb)
    norm = math.sqrt(sum(v ** 2 for v in ba.values())) * \
           math.sqrt(sum(v ** 2 for v in bb.values()))
    return dot / (norm + 1e-9)
def vec_cosine(a, b, vec) -> float:
    mats = vec.transform([str(a), str(b)])
    return float(_sk_cos(mats[0], mats[1])[0][0])
def build_model_a_dataset(*_, **__):
    raise NotImplementedError("Provide preprocessing.py")
def apply_vectorizer(texts, vec):
    return vec.transform(texts)
def make_sample_string(article, question, option):
        return normalize(f"{article} {question} {option}")

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------

# Local paths (commented)
# _MODEL_DEST   = os.path.join(_THIS_DIR, "..", "models", "model_a", "traditional")
# _REPORTS_DEST = os.path.join(ARTIFACT_DIR, "reports")

# Kaggle paths (active)
_MODEL_DEST   = os.path.join(BASE_PROJ, "models", "model_a", "neural")
_REPORTS_DEST = os.path.join(ARTIFACT_DIR, "reports")
os.makedirs(_MODEL_DEST,   exist_ok=True)
os.makedirs(_REPORTS_DEST, exist_ok=True)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

_CHOICES    = ["A", "B", "C", "D"]
_RNG_SEED   = 42

# ---------------------------------------------------------------------------
# SECTION 1 — Data loading
# ---------------------------------------------------------------------------

def _fetch_arrays():
    """Load OHE feature matrices, labels, and handcrafted features."""
    d = ARTIFACT_DIR
    files = [
        "X_train_ohe.npz", "X_val_ohe.npz", "X_test_ohe.npz",
        "y_train.npy", "y_val.npy", "y_test.npy",
        "hc_train.npy", "hc_val.npy", "hc_test.npy",
    ]
    loaders = [load_npz if f.endswith(".npz") else np.load for f in files]
    results = []
    for fname, loader in tqdm(zip(files, loaders), total=len(files), desc="loading arrays", unit="file"):
        results.append(loader(os.path.join(d, fname)))
    return tuple(results)


# keep old name available
load_processed_data = _fetch_arrays


def _integrity_check(X_tr, y_tr, X_va, y_va) -> dict:
    assert X_tr.shape[0] == len(y_tr), "Train X/y mismatch"
    assert X_va.shape[0] == len(y_va), "Val X/y mismatch"
    return {
        "train_rows":      X_tr.shape[0],
        "val_rows":        X_va.shape[0],
        "train_pos_frac":  float(np.mean(y_tr)),
        "val_pos_frac":    float(np.mean(y_va)),
    }


# ---------------------------------------------------------------------------
# SECTION 2 — Internal matrix helpers
# ---------------------------------------------------------------------------




# ---------------------------------------------------------------------------
# SECTION 3 — Neural network classifier (replaces LR + SVM)
# ---------------------------------------------------------------------------

def _stack_hc(X, hc):
    """Stack handcrafted dense features onto sparse OHE matrix."""
    if hc is None or hc.shape[1] == 0:
        return X
    return sparse_hstack([X, hc.astype(np.float32)], format="csr")


def fit_answer_verifier(X_tr, y_tr, X_va=None, y_va=None, hc_tr=None, hc_va=None, **kw) -> AnswerVerifier:
    """
    Train a feed-forward neural network on OHE + handcrafted features.
    Loss is weighted by the inverse class frequency to handle
    the ≈3:1 wrong-to-correct ratio.  Early stopping on validation loss.
    """
    print("[A] NN", end="", flush=True)
    X_tr = _stack_hc(X_tr, hc_tr)
    if X_va is not None:
        X_va = _stack_hc(X_va, hc_va)
    input_dim = X_tr.shape[1]
    pos = y_tr.sum()
    neg = len(y_tr) - pos
    pos_weight = neg / max(pos, 1)          # ≈3.0 for the RACE dataset

    model = AnswerVerifier(
        input_dim=input_dim,
        hidden_dims=kw.get("hidden_dims", [512, 128]),
        dropout=kw.get("dropout", 0.3),
    )
    model = train_nn(
        model,
        X_tr, y_tr,
        X_val=X_va, y_val=y_va,
        epochs=kw.get("epochs", 30),
        batch_size=kw.get("batch_size", 256),
        lr=kw.get("lr", 1e-3),
        weight_decay=kw.get("weight_decay", 1e-4),
        pos_weight=pos_weight,
        patience=kw.get("patience", 5),
        verbose=True,
    )
    print(" ✓")
    return model

# compat alias
train_answer_verifier = fit_answer_verifier





# ---------------------------------------------------------------------------
# SECTION 7 — Cosine-similarity accuracy
# ---------------------------------------------------------------------------

def _sentence_max_cosine(article: str, option_text: str, vec) -> float:
    """Max TF-IDF cosine over all article sentences vs. option_text."""
    sents = sentence_fragments(str(article))
    if not sents:
        return vec_cosine(article, option_text, vec)
    s_vecs = vec.transform(sents)
    o_vec  = vec.transform([str(option_text)])
    return float(_sk_cos(s_vecs, o_vec).flatten().max())


def retrieval_accuracy(
    tfidf_vec,
    df: pd.DataFrame,
    n_rows: Optional[int] = None,
    ohe_vec=None,
    sentence_level: bool = True,
    alpha: float = 0.7,
) -> dict:
    """
    For each row select the option with the highest article–option similarity;
    report accuracy, average correct-option sim, average wrong-option sim, gap.
    """
    if n_rows and len(df) > n_rows:
        df = df.sample(n_rows, random_state=_RNG_SEED)

    n_hit, correct_sims, wrong_sims = 0, [], []

    for _, row in df.iterrows():
        article    = str(row["article"])
        gold       = str(row["answer"]).strip().upper()
        options    = {o: str(row[o]) for o in _CHOICES}
        per_option: Dict[str, float] = {}

        for key, txt in options.items():
            if tfidf_vec is not None:
                t_score = (
                    _sentence_max_cosine(article, txt, tfidf_vec)
                    if sentence_level
                    else vec_cosine(article, txt, tfidf_vec)
                )
            else:
                t_score = 0.0

            if ohe_vec is not None:
                o_score = (
                    _sentence_max_cosine(article, txt, ohe_vec)
                    if sentence_level
                    else vec_cosine(article, txt, ohe_vec)
                )
                per_option[key] = alpha * t_score + (1.0 - alpha) * o_score
            else:
                per_option[key] = t_score

        predicted = max(per_option, key=per_option.get)
        if predicted == gold:
            n_hit += 1
        correct_sims.append(per_option[gold])
        wrong_sims.extend(v for k, v in per_option.items() if k != gold)

    total = len(df)
    return {
        "accuracy":        n_hit / total if total else 0.0,
        "avg_correct_sim": float(np.mean(correct_sims)) if correct_sims else 0.0,
        "avg_wrong_sim":   float(np.mean(wrong_sims))   if wrong_sims   else 0.0,
        "sim_gap": float(np.mean(correct_sims) - np.mean(wrong_sims))
                   if (correct_sims and wrong_sims) else 0.0,
    }

# compat alias
cosine_similarity_accuracy = retrieval_accuracy


def sweep_retrieval_params(tfidf_vec, ohe_vec, eval_df: pd.DataFrame) -> dict:
    """Grid-search sentence_level × alpha; return best-accuracy configuration."""
    grid = [(sl, a) for sl in (True, False) for a in (0.55, 0.70, 0.85)]
    champion: dict = {"accuracy": -1.0}
    for sent_lvl, a in grid:
        result = retrieval_accuracy(
            tfidf_vec, eval_df,
            ohe_vec=ohe_vec,
            sentence_level=sent_lvl,
            alpha=a,
        )
        if result["accuracy"] > champion["accuracy"]:
            champion = {**result, "use_sentence_max": sent_lvl, "alpha": a}
    return champion

# compat alias
tune_cosine_similarity = sweep_retrieval_params


def domain_overlap(
    train_df: pd.DataFrame,
    test_df:  pd.DataFrame,
    tfidf_vec,
    n_sample: int = 200,
) -> float:
    """
    Average max-cosine similarity from each test article to the nearest
    training article — a proxy for train/test domain overlap.
    """
    tr_corpus = train_df["article"].dropna().sample(
        min(n_sample, len(train_df)), random_state=_RNG_SEED).tolist()
    te_corpus = test_df["article"].dropna().sample(
        min(n_sample, len(test_df)),  random_state=_RNG_SEED).tolist()

    tr_vecs = tfidf_vec.transform(tr_corpus)
    te_vecs = tfidf_vec.transform(te_corpus)

    block, max_sims = 50, []
    for i in tqdm(range(0, len(te_corpus), block), desc="domain overlap", unit="chunk"):
        chunk   = te_vecs[i:i + block]
        sim_mat = _sk_cos(chunk, tr_vecs)
        max_sims.extend(sim_mat.max(axis=1).tolist())

    return float(np.mean(max_sims))

# compat alias
compute_train_test_domain_similarity = domain_overlap


# ---------------------------------------------------------------------------
# SECTION 8 — 4-way MCQ accuracy
# ---------------------------------------------------------------------------

def _build_hc_row(article, question, opt_text, tfidf_vec, idf_dict=None, vocab=None):
    """Compute the 12 handcrafted features for one (article, question, option) triple."""
    if tfidf_vec is not None and len(opt_text) > 0:
        art_vec = tfidf_vec.transform([article])
        q_vec = tfidf_vec.transform([question])
        opt_vec = tfidf_vec.transform([opt_text])
        sim_ao = float(_sk_cos(art_vec, opt_vec)[0, 0])
        sim_qo = float(_sk_cos(q_vec, opt_vec)[0, 0])
        opt_tfidf_max = float(opt_vec.max())
    else:
        sim_ao = sim_qo = 0.0
        opt_tfidf_max = 0.0

    art_tokens = set(article.split())
    opt_tokens = set(opt_text.split())
    q_tokens = set(question.split())
    overlap_ao = len(opt_tokens & art_tokens) / (len(opt_tokens) + 1e-9)
    overlap_qo = len(opt_tokens & q_tokens) / (len(opt_tokens) + 1e-9)
    opt_len_ratio = min(len(opt_text) / 200.0, 1.0)
    overlap_oa = len(opt_tokens & art_tokens) / (len(art_tokens) + 1e-9)
    overlap_oq = len(opt_tokens & q_tokens) / (len(q_tokens) + 1e-9)

    opt_idf_mean = 0.0
    if idf_dict and vocab and opt_tokens:
        idfs = [idf_dict.get(w, 0.0) for w in opt_tokens if w in vocab]
        opt_idf_mean = sum(idfs) / len(idfs) if idfs else 0.0

    has_digit = 1.0 if re.search(r"\d", opt_text) else 0.0

    art_bigrams = set(
        " ".join(article.split()[i:i+2])
        for i in range(len(article.split()) - 1)
    )
    opt_bigrams = set(
        " ".join(opt_text.split()[i:i+2])
        for i in range(len(opt_text.split()) - 1)
    )
    shared_bigrams = len(art_bigrams & opt_bigrams) / (len(opt_bigrams) + 1e-9) if opt_bigrams else 0.0

    starts_cap = 1.0 if opt_text and opt_text[0].isupper() else 0.0

    return [sim_ao, sim_qo, overlap_ao, overlap_qo, opt_len_ratio,
            overlap_oa, overlap_oq, opt_idf_mean, has_digit,
            shared_bigrams, opt_tfidf_max, starts_cap]


def mcq_accuracy(clf, ohe_vec, df: pd.DataFrame, n_rows: Optional[int] = None, tfidf_vec=None) -> float:
    """
    For each row score all four options; pick the one with highest P(correct).
    Returns fraction of rows where the predicted option matches the gold label.
    Uses handcrafted features when tfidf_vec is provided.
    """
    if n_rows and len(df) > n_rows:
        df = df.sample(n_rows, random_state=_RNG_SEED)
    hits = total = 0
    use_hc = tfidf_vec is not None
    idf_dict = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_vec.idf_)) if use_hc else None
    vocab = set(tfidf_vec.get_feature_names_out()) if use_hc else None
    for _, row in df.iterrows():
        article = clean_text(row["article"])
        question = clean_text(row["question"])
        encoded = []
        hc_rows = []
        for o in _CHOICES:
            opt_text = clean_text(str(row[o]))
            encoded.append(make_sample_string(article, question, opt_text))
            if use_hc:
                hc_rows.append(
                    _build_hc_row(article, question, opt_text, tfidf_vec, idf_dict, vocab)
                )
        X = ohe_vec.transform(encoded)
        if use_hc and hc_rows:
            hc = np.array(hc_rows, dtype=np.float32)
            X = _stack_hc(X, hc)
        probs = clf.predict_proba(X)[:, 1]
        best = _CHOICES[int(np.argmax(probs))]
        if best == str(row["answer"]).strip().upper():
            hits += 1
        total += 1
    return hits / total if total else 0.0

# compat alias
compute_4way_accuracy = mcq_accuracy


# ---------------------------------------------------------------------------
# SECTION 9 — Binary evaluation
# ---------------------------------------------------------------------------

def score_classifier(clf, X, y, tag: str = "") -> dict:
    preds = clf.predict(X)
    acc   = accuracy_score(y, preds)
    prec  = precision_score(y, preds, average="macro", zero_division=0)
    rec   = recall_score(y,    preds, average="macro", zero_division=0)
    f1    = f1_score(y,        preds, average="macro", zero_division=0)
    em    = float(np.mean([str(p) == str(t) for p, t in zip(preds, y)]))
    if tag:
        print(f"\n  {tag}")
        for name, val in [("acc", acc), ("prec", prec), ("rec", rec), ("f1", f1), ("em", em)]:
            print(f"    {name}={val:.4f}")
    return {"accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "exact_match": em, "predictions": preds}

# compat alias
evaluate_binary = score_classifier


# ---------------------------------------------------------------------------
# SECTION 10 — MCQ generation
# ---------------------------------------------------------------------------

_WH_BANK: Dict[str, List[str]] = {
    "what": [
        "What does the passage say about {topic}?",
        "What is {topic} according to the passage?",
        "What role does {topic} play in the passage?",
    ],
    "how": [
        "How is {topic} described in the passage?",
        "How does {topic} relate to the main idea?",
    ],
    "why": [
        "Why is {topic} important according to the passage?",
        "Why is {topic} mentioned in the passage?",
    ],
    "where": ["Where does {topic} occur according to the passage?"],
    "when":  ["When is {topic} relevant in the context of the passage?"],
    "who":   ["Who is associated with {topic} in the passage?"],
}

_ALL_TEMPLATES: List[str] = list(chain.from_iterable(_WH_BANK.values()))

_GEN_STOP = frozenset({
    "a","an","the","is","it","in","of","to","and","or","for","on","with","as",
    "at","by","be","was","are","were","this","that","from","but","not","have",
    "has","had","he","she","they","we","you","i","do","did","will","its",
    "their","which","who","what","how","when","where","there","these","those",
    "can","could","would","should","also","been","being",
})


def _content_words(text: str) -> List[str]:
    return [t for t in word_tokens(text) if t not in _GEN_STOP and len(t) >= 4]


def _sent_score(sentence: str, answer_vocab: set) -> float:
    s_vocab = set(_content_words(sentence))
    return len(answer_vocab & s_vocab) + len(sentence.split()) / 100.0


def _drop_short_sents(sents: List[str], min_len: int = 10) -> List[str]:
    return [s for s in sents if len(s) >= min_len]


def extract_candidate_sentences(
    article: str, anchor: str, top_k: int = 5
) -> List[Tuple[str, float]]:
    """Score each article sentence by keyword overlap with anchor text."""
    raw   = sentence_fragments(article)
    sents = _drop_short_sents(raw)
    if not sents:
        return [(article[:200], 1.0)]
    a_vocab = set(_content_words(anchor))
    ranked  = [(s, _sent_score(s, a_vocab)) for s in sents]
    ranked.sort(key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


def _pick_topic_word(sentence: str, used: set) -> str:
    pool = [t for t in _content_words(sentence) if t not in used]
    if not pool:
        pool = _content_words(sentence)
    return max(pool, key=len) if pool else "this topic"


def _extract_answer_span(sentence: str, question: str, max_words: int = 8) -> str:
    words  = str(sentence).split()
    if not words:
        return sentence[:40]
    q_set  = set(normalize(question).split())
    anchor = next((i for i, w in enumerate(words) if normalize(w) in q_set), None)
    if anchor is None:
        cands = [
            (sum(1 for w in words[s:s + max_words] if w.lower() not in _GEN_STOP), s)
            for s in range(max(0, len(words) - max_words + 1))
        ]
        _, best_start = max(cands) if cands else (0, 0)
        return " ".join(words[best_start:best_start + max_words])
    start = max(0, anchor - max_words // 2)
    end   = min(len(words), start + max_words)
    start = max(0, end - max_words)
    return " ".join(words[start:end])


# -- Question ranker features --

def _ranker_features(question: str, source_sent: str, article: str) -> np.ndarray:
    q_tok = word_tokens(question)
    s_tok = word_tokens(source_sent)
    a_tok = word_tokens(article)
    q_len = len(q_tok)
    return np.array([
        len(set(q_tok) & set(s_tok)) / (len(set(q_tok)) + 1e-9),
        len(set(q_tok) & set(a_tok)) / (len(set(q_tok)) + 1e-9),
        q_len,
        float(question.split()[0].lower() in {"what","who","where","when","why","how"})
            if question else 0.0,
        float(question.strip().endswith("?")),
        float(any(t.endswith(("ed","ing","es","tion")) for t in q_tok)),
        len(set(q_tok)) / (q_len + 1e-9),
        len(_content_words(question)) / (q_len + 1e-9),
    ], dtype=np.float32)


def _heuristic_rank(pairs: List[Tuple[str, str]], article: str) -> List[Tuple[str, str]]:
    art_vocab = set(word_tokens(article))
    def score(q, _s):
        return len(set(word_tokens(q)) & art_vocab) / (len(word_tokens(q)) + 1e-9)
    return sorted(pairs, key=lambda p: score(*p), reverse=True)


# -- Distractor assembly --

def _build_distractors(article: str, correct: str, n: int = 3) -> List[str]:
    """
    Build plausible distractors in priority order:
      1. Article sentence fragments with moderate article-answer similarity
      2. Keyword-substituted answer variants
    """
    sents       = sentence_fragments(article)
    ans_vocab   = set(word_tokens(correct))
    art_content = _content_words(article)

    candidates: List[Tuple[float, str]] = []
    for sent in sents:
        phrase = _extract_answer_span(sent, correct, max_words=6)
        if set(word_tokens(phrase)) == ans_vocab or len(phrase) <= 5:
            continue
        sim = dot_cosine(phrase, correct)
        candidates.append((sim, phrase))

    candidates.sort(reverse=True)
    chosen = [ph for sim, ph in candidates if 0.08 < sim < 0.85][:n]

    swap_pool  = list(set(art_content) - ans_vocab)
    ans_words  = correct.split()
    if len(swap_pool) >= 2 and len(ans_words) >= 2:
        for _ in range(n + 1):
            variant = ans_words[:]
            variant[random.randrange(len(variant))] = random.choice(swap_pool)
            chosen.append(" ".join(variant))

    seen, unique = set(), []
    for d in chosen:
        key = normalize(d)
        if key not in seen and key != normalize(correct):
            seen.add(key)
            unique.append(d)
        if len(unique) >= n:
            break

    while len(unique) < n:
        unique.append(f"none of the above ({len(unique) + 1})")

    return unique[:n]

# compat alias
generate_distractors = _build_distractors


def compose_questions(
    article: str,
    count: int = 5,
    ranker=None,
) -> List[dict]:
    """
    Three-phase MCQ generation:
      Phase 1 — extract candidate sentences by keyword-overlap scoring
      Phase 2 — apply Wh-word templates over topic keywords
      Phase 3 — rank with ML ranker (if available) or heuristic fallback

    Each returned dict contains:
      question, answer, correct_letter, distractors, source_sentence, options
    """
    sents = sentence_fragments(article)
    if not sents:
        sents = [article[:200]]

    richest  = max(sents, key=lambda s: len(_content_words(s)))
    a_vocab  = set(_content_words(richest))

    candidates = extract_candidate_sentences(
        article, richest, top_k=max(count * 2, 10)
    )

    templates = _ALL_TEMPLATES[:]
    random.shuffle(templates)

    raw_pairs: List[Tuple[str, str]] = []
    for i, (sent, _) in enumerate(candidates):
        topic    = _pick_topic_word(sent, a_vocab)
        template = templates[i % len(templates)]
        raw_pairs.append((template.format(topic=topic), sent))

    if ranker is not None:
        feats = np.array([_ranker_features(q, s, article) for q, s in raw_pairs])
        try:
            scores = ranker.predict_proba(feats)[:, 1]
            ranked = [p for _, p in sorted(zip(scores, raw_pairs), reverse=True)]
        except Exception:
            ranked = _heuristic_rank(raw_pairs, article)
    else:
        ranked = _heuristic_rank(raw_pairs, article)

    results, seen_keys = [], set()
    for question, src_sent in ranked:
        answer  = _extract_answer_span(src_sent, question, max_words=8)
        key     = normalize(answer)[:20]
        if key in seen_keys:
            continue
        seen_keys.add(key)

        distractors = _build_distractors(article, answer, n=3)
        opts_list   = [answer] + distractors
        random.shuffle(opts_list)
        opts_dict   = dict(zip(_CHOICES, opts_list))
        correct_ltr = next(k for k, v in opts_dict.items() if v == answer)

        results.append({
            "question":        question,
            "answer":          answer,
            "correct_letter":  correct_ltr,
            "distractors":     distractors,
            "source_sentence": src_sent,
            "options":         opts_dict,
        })
        if len(results) >= count:
            break

    if not results:
        q = "What is the main idea of the passage?"
        a = _extract_answer_span(richest, q, max_words=8)
        d = _build_distractors(article, a, n=3)
        o = dict(zip(_CHOICES, [a] + d))
        results.append({
            "question": q, "answer": a, "correct_letter": "A",
            "distractors": d, "source_sentence": richest, "options": o,
        })

    return results

# compat alias
generate_questions_from_passage = compose_questions


# ---------------------------------------------------------------------------
# SECTION 11 — Generation metrics (BLEU / ROUGE / METEOR)
# ---------------------------------------------------------------------------

def _lex_tokens(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", text.lower())


def _ngram_freq(tokens: List[str], n: int) -> Counter:
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu(reference: str, hypothesis: str, max_n: int = 4) -> float:
    """
    Sentence-level BLEU with modified n-gram precision + brevity penalty.
    Near-zero precisions are smoothed with 1e-9 to avoid -inf log.
    """
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    precisions = []
    for n in range(1, max_n + 1):
        h_ng = _ngram_freq(hyp, n)
        r_ng = _ngram_freq(ref, n)
        if not h_ng:
            precisions.append(0.0)
            continue
        clip = sum(min(cnt, r_ng[ng]) for ng, cnt in h_ng.items())
        precisions.append(clip / sum(h_ng.values()))
    bp       = 1.0 if len(hyp) >= len(ref) else math.exp(1 - len(ref) / len(hyp))
    smoothed = [p if p > 0 else 1e-9 for p in precisions]
    return bp * math.exp(sum(math.log(p) for p in smoothed) / max_n)

# compat alias
sentence_bleu_score = bleu


def rouge(reference: str, hypothesis: str) -> dict:
    """ROUGE-1, ROUGE-2, and ROUGE-L F1 scores."""
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)

    def _f1_ngram(r, h, n):
        rng = _ngram_freq(r, n)
        hng = _ngram_freq(h, n)
        if not rng or not hng:
            return 0.0
        shared = sum(min(rng[k], hng[k]) for k in rng if k in hng)
        prec   = shared / sum(hng.values())
        rec    = shared / sum(rng.values())
        return 2 * prec * rec / (prec + rec + 1e-9)

    def _lcs_len(a, b):
        m, n = len(a), len(b)
        dp   = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                dp[i][j] = (
                    dp[i-1][j-1] + 1
                    if a[i-1] == b[j-1]
                    else max(dp[i-1][j], dp[i][j-1])
                )
        return dp[m][n]

    r1  = _f1_ngram(ref, hyp, 1)
    r2  = _f1_ngram(ref, hyp, 2)
    lcs = _lcs_len(ref, hyp)
    if ref and hyp:
        p_l = lcs / len(hyp)
        r_l = lcs / len(ref)
        rl  = 2 * p_l * r_l / (p_l + r_l + 1e-9)
    else:
        rl  = 0.0
    return {"rouge1_f": r1, "rouge2_f": r2, "rougeL_f": rl}

# compat alias
rouge_scores = rouge


def meteor(reference: str, hypothesis: str) -> float:
    """Simplified METEOR: precision/recall harmonic mean with fragmentation penalty."""
    ref = _lex_tokens(reference)
    hyp = _lex_tokens(hypothesis)
    if not ref or not hyp:
        return 0.0
    rc  = Counter(ref)
    hc  = Counter(hyp)
    m   = sum(min(rc[t], hc[t]) for t in rc if t in hc)
    if m == 0:
        return 0.0
    prec   = m / len(hyp)
    rec    = m / len(ref)
    fmean  = (10 * prec * rec) / (9 * prec + rec + 1e-9)
    return fmean * (1 - 0.5 / max(m, 1))

# compat alias
meteor_score = meteor


def generation_metrics(
    generated: List[dict],
    df: pd.DataFrame,
    n_sample: int = 300,
) -> dict:
    """Compare generated answers to source sentences; return aggregate scores."""
    if not generated:
        return {}
    if len(df) > n_sample:
        df = df.sample(n_sample, random_state=_RNG_SEED)

    b_vals, r1_vals, r2_vals, rl_vals, m_vals = [], [], [], [], []
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="generation metrics", unit="row")):
        sample  = generated[i % len(generated)]
        ref     = sample["source_sentence"]
        hyp     = sample["answer"]
        b_vals.append(bleu(ref, hyp))
        rg = rouge(ref, hyp)
        r1_vals.append(rg["rouge1_f"])
        r2_vals.append(rg["rouge2_f"])
        rl_vals.append(rg["rougeL_f"])
        m_vals.append(meteor(ref, hyp))

    return {
        "bleu":      float(np.mean(b_vals)),
        "rouge1_f":  float(np.mean(r1_vals)),
        "rouge2_f":  float(np.mean(r2_vals)),
        "rougeL_f":  float(np.mean(rl_vals)),
        "meteor":    float(np.mean(m_vals)),
        "n_samples": len(b_vals),
    }

# compat alias
compute_generation_metrics = generation_metrics


# ---------------------------------------------------------------------------
# SECTION 12 — Persistence
# ---------------------------------------------------------------------------

def persist_models(nn_model, metrics: dict):
    """Write all Model-A artifacts to _MODEL_DEST."""
    save_checkpoint(nn_model, os.path.join(_MODEL_DEST, "answer_verifier.pt"))
    joblib.dump(metrics, os.path.join(_MODEL_DEST, "metrics.pkl"))
    print(f"\n  artifacts → {_MODEL_DEST}")
    _export_metrics_to_csv(metrics)


def _export_metrics_to_csv(metrics: dict):
    """Export all metrics to CSV files in the reports directory."""
    import os
    os.makedirs(_REPORTS_DEST, exist_ok=True)

    # 1. NN binary classification metrics
    if "nn" in metrics and metrics["nn"]:
        nn_data = [{"model": "NN"}]
        nn_data[0].update(
            {k: v for k, v in metrics["nn"].items() if k != "predictions"}
        )
        df_nn = pd.DataFrame(nn_data)
        csv_path = os.path.join(_REPORTS_DEST, "model_a_binary_metrics.csv")
        df_nn.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")

    # 2. Cosine retrieval metrics
    if "cosine_retrieval" in metrics and metrics["cosine_retrieval"]:
        df_cosine = pd.DataFrame([metrics["cosine_retrieval"]])
        csv_path = os.path.join(_REPORTS_DEST, "model_a_cosine_retrieval_metrics.csv")
        df_cosine.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")

    # 3. Text generation metrics
    if "text_generation" in metrics and metrics["text_generation"]:
        df_gen = pd.DataFrame([metrics["text_generation"]])
        csv_path = os.path.join(_REPORTS_DEST, "model_a_text_generation_metrics.csv")
        df_gen.to_csv(csv_path, index=False)
        print(f"    → {csv_path}")

# compat alias
save_models = persist_models


def _load_artifact(fname: str):
    path = os.path.join(_MODEL_DEST, fname)
    return joblib.load(path) if os.path.exists(path) else None


# ---------------------------------------------------------------------------
# SECTION 13 — Main training pipeline
# ---------------------------------------------------------------------------

def _train_transformer_av(train_csv, val_csv):
    """Optional transformer answer verifier training if transformers is installed."""
    try:
        from src.transformer_train import train_answer_verifier_transformer
    except ImportError:
        print("  transformers not installed — skipping")
        return
    if not os.path.exists(train_csv):
        print(f"  {train_csv} not found — skipping transformer AV")
        return
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv) if os.path.exists(val_csv) else None
    try:
        train_answer_verifier_transformer(train_df, val_df, epochs=5, use_lora=True)
    except Exception as e:
        print(f"  transformer AV failed: {e}")


def train_all():
    """
    End-to-end Model-A pipeline (neural network + transformer):
      1  Load OHE feature arrays
      2  Train NN answer verifier
      3  Optional: K-Means / Label Propagation analysis
      4  Evaluate on validation: binary metrics, 4-way MCQ, cosine accuracy
      5  Generate MCQs and score BLEU / ROUGE / METEOR
      6  Train transformer answer verifier (optional)
      7  Final evaluation on held-out test split
      8  Persist all artifacts
    """
    BAR = "─" * 58
    print(BAR)
    print("  verifier_a  ::  neural network training run")
    print(BAR)

    # -- 1. Load arrays --
    print("\n(1) loading feature arrays")
    X_tr, X_va, X_te, y_tr, y_va, y_te, hc_tr, hc_va, hc_te = _fetch_arrays()
    info = _integrity_check(X_tr, y_tr, X_va, y_va)
    print(f"    train={X_tr.shape[0]:,}  val={X_va.shape[0]:,}  test={X_te.shape[0]:,}")
    print(f"    features={X_tr.shape[1]:,}")
    print(f"    pos-rate — train={info['train_pos_frac']:.3f}  val={info['val_pos_frac']:.3f}")

    # -- 2. Train neural network answer verifier --
    print("\n(2) training neural network answer verifier")
    nn_model = fit_answer_verifier(X_tr, y_tr, X_va, y_va, hc_tr=hc_tr, hc_va=hc_va, epochs=30)

    # -- 3. Validation snapshot --
    print("\n(3) validation snapshot")
    X_va_stacked = _stack_hc(X_va, hc_va)
    nn_res = score_classifier(nn_model, X_va_stacked, y_va, "NN (val)")

    # Data / vectorizer paths
    ohe_path   = os.path.join(ARTIFACT_DIR, "ohe_vectorizer.pkl")
    tfidf_path = os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.pkl")
    val_csv    = os.path.join(ARTIFACT_DIR, "val_clean.csv")
    train_csv  = os.path.join(ARTIFACT_DIR, "train_clean.csv")
    test_csv   = os.path.join(ARTIFACT_DIR, "test_clean.csv")

    cos_metrics, nn_4w = {}, 0.0
    ohe_vec = tfidf_vec = None

    if all(os.path.exists(p) for p in [ohe_path, tfidf_path, val_csv]):
        with open(ohe_path,   "rb") as fh: ohe_vec   = pickle.load(fh)
        with open(tfidf_path, "rb") as fh: tfidf_vec = pickle.load(fh)

        val_df  = pd.read_csv(val_csv)
        eval_df = val_df.sample(min(1_000, len(val_df)), random_state=_RNG_SEED)

        print("\n  4-way MCQ accuracy (val):")
        nn_4w = mcq_accuracy(nn_model, ohe_vec, eval_df, tfidf_vec=tfidf_vec)
        print(f"    NN : {nn_4w:.4f}")

        print("\n  cosine-similarity retrieval accuracy (val):")
        best = sweep_retrieval_params(tfidf_vec, ohe_vec, eval_df)
        print(f"    sent_level={best['use_sentence_max']}  alpha={best['alpha']}")
        print(f"    acc={best['accuracy']:.4f}  "
              f"avg_correct={best['avg_correct_sim']:.4f}  "
              f"avg_wrong={best['avg_wrong_sim']:.4f}  "
              f"gap={best['sim_gap']:.4f}")
        cos_metrics = dict(best)

        if os.path.exists(train_csv) and os.path.exists(test_csv):
            tr_df = pd.read_csv(train_csv)
            te_df = pd.read_csv(test_csv)
            dom   = domain_overlap(tr_df, te_df, tfidf_vec, n_sample=400)
            print(f"\n  domain overlap (train↔test): {dom:.4f}")
            cos_metrics["domain_similarity"] = dom

    # -- 4. Question generation + text metrics --
    print("\n(4) MCQ generation checks")
    gen_m: dict = {}
    if os.path.exists(val_csv):
        gen_df  = pd.read_csv(val_csv)
        sub_df  = gen_df.sample(min(400, len(gen_df)), random_state=_RNG_SEED)
        all_gen = []
        for _, row in tqdm(sub_df.iterrows(), total=len(sub_df), desc="generating MCQs", unit="article"):
            art = str(row.get("article", ""))
            if len(art) >= 50:
                all_gen.extend(compose_questions(art, count=3))
        if all_gen:
            gen_m = generation_metrics(all_gen, sub_df)
            print(f"    generated {len(all_gen)} question-answer pairs")
            for k in ("bleu", "rouge1_f", "rouge2_f", "rougeL_f", "meteor"):
                print(f"    {k.upper():<12}: {gen_m.get(k, 0):.4f}")
            report = os.path.join(_REPORTS_DEST, "generation_metrics.pkl")
            joblib.dump(gen_m, report)
            print(f"    metrics → {report}")
        else:
            print("    (no usable articles — skipping generation eval)")
    else:
        print("    (val_clean.csv not found — skipping generation eval)")

    # -- 5. Transformer answer verifier (optional) --
    print("\n(5) transformer answer verifier")
    _train_transformer_av(train_csv, val_csv)

    # -- 6. Test evaluation --
    print("\n(6) held-out test split")
    X_te_stacked = _stack_hc(X_te, hc_te)
    score_classifier(nn_model, X_te_stacked, y_te, "NN (test)")
    preds = nn_model.predict(X_te_stacked)
    te_a  = accuracy_score(y_te, preds)
    te_f  = f1_score(y_te, preds, average="macro", zero_division=0)
    te_e  = float(np.mean([str(p) == str(t) for p, t in zip(preds, y_te)]))
    print(f"\n  NN (test)\n    acc={te_a:.4f}  f1={te_f:.4f}  em={te_e:.4f}")

    # -- Aggregate metrics dict --
    metrics = {
        "nn":               {**(nn_res or {}), "4way_acc": nn_4w},
        "cosine_retrieval":  cos_metrics,
        "text_generation":   gen_m,
    }

    persist_models(nn_model, metrics)
    print("\n" + "=" * 65)
    print("  verifier_a training complete")
    print(BAR)
    return nn_model, metrics


# ---------------------------------------------------------------------------
# Public inference API
# ---------------------------------------------------------------------------

def load_model_a() -> dict:
    """Load the NN answer verifier and metrics from disk."""
    nn_path = os.path.join(_MODEL_DEST, "answer_verifier.pt")
    if not os.path.exists(nn_path):
        print(f"  WARNING: NN checkpoint not found at {nn_path}")
        return {"model": None, "metrics": None}
    input_dim = (  # try to infer from saved meta
        _load_checkpoint_meta(nn_path, "input_dim") or 10_000
    )
    model = AnswerVerifier(input_dim=input_dim)
    ckpt = torch.load(nn_path, map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    metrics = _load_artifact("metrics.pkl")
    return {"model": model, "metrics": metrics}


def _load_checkpoint_meta(path: str, key: str):
    """Read a single meta key from a saved NN checkpoint."""
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=True)
        return ckpt.get("meta", {}).get(key)
    except Exception:
        return None


def verify_answer(
    article: str,
    question: str,
    option: str,
    models: dict,
    ohe_vec,
) -> dict:
    """
    Predict whether `option` is the correct answer using the neural network.
    Returns prediction, probability, and per-model fields for backwards compat.
    """
    sample  = ohe_vec.transform([make_sample_string(article, question, option)])
    nn_prob = float(models["model"].predict_proba(sample)[0, 1])

    return {
        "prediction":     int(nn_prob > 0.5),
        "probability":    nn_prob,
        "lr_proba":       nn_prob,
        "svm_proba":      nn_prob,
        "soft_ensemble":  nn_prob,
        "stack_ensemble": nn_prob,
        "nn_proba":       nn_prob,
    }


def generate_mcq(article: str, n_questions: int = 5) -> List[dict]:
    """Public API: generate n_questions MCQs from a passage."""
    return compose_questions(str(article), count=n_questions)


# ---------------------------------------------------------------------------
# Optional: transformer-based model loading (only if transformers is installed)
# ---------------------------------------------------------------------------

def _try_import_transformers():
    try:
        import transformers as _tf
        import peft as _peft
        return True
    except ImportError:
        return False


def load_transformer_answer_verifier():
    """Load a trained TransformerAnswerVerifier checkpoint + tokenizer."""
    if not _try_import_transformers():
        print("  transformers/peft not installed. Skipping transformer model.")
        return None, None

    from transformers import AutoTokenizer
    from src.nn_models import TransformerAnswerVerifier

    meta_path = os.path.join(_MODEL_DEST.replace("neural", "transformer"), "transformer_meta.json")
    ckpt_path = os.path.join(_MODEL_DEST.replace("neural", "transformer"), "answer_verifier_transformer.pt")

    if not os.path.exists(ckpt_path):
        return None, None

    with open(meta_path) as f:
        import json
        meta = json.load(f)

    model_name = meta.get("model_name", "bert-base-uncased")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = TransformerAnswerVerifier(model_name=model_name, num_labels=2)
    model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=True))
    model.eval()
    print(f"  loaded transformer answer verifier ({model_name})")
    return model, tokenizer


def verify_answer_transformer(
    article: str,
    question: str,
    option: str,
    model,
    tokenizer,
) -> dict:
    """Predict P(correct) using the transformer answer verifier."""
    text = f"article: {article}\nquestion: {question}\noption: {option}"
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=384, return_tensors="pt")
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
        probs = torch.softmax(out.logits, dim=-1)
        prob_correct = float(probs[0, 1])
    return {
        "prediction": int(prob_correct > 0.5),
        "probability": prob_correct,
        "nn_proba": prob_correct,
    }


# ---------------------------------------------------------------------------
if __name__ == "__main__":
    train_all()

## MODEL B TRAINING

In [ ]:
"""
scorer_b.py  —  distractor & hint scorers (batched TF-IDF, logistic regression)
"""
from __future__ import annotations

import os, pickle, sys
from collections import Counter
from typing import List, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# from src.nn_models import (
#     DistractorScorer,
#     HintScorer,
#     DEVICE,
#     load_checkpoint,
#     save_checkpoint,
#     train_nn,
# )

# from preprocessing import (
#     ARTIFACT_DIR,
#     sentence_fragments,
#     word_tokens,
#     normalize,
# )

# ---------------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------------

# Local development paths (commented)
# _THIS_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else "/kaggle/working/data"
# ARTIFACT_DIR = os.path.normpath(os.path.join(_THIS_DIR, "..", "data", "processed"))
# _SCORER_OUT = os.path.normpath(os.path.join(_THIS_DIR, "..", "models", "model_b", "traditional"))

# Local paths
try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    BASE_PROJ  = os.path.normpath(os.path.join(_THIS_DIR, ".."))
except NameError:
    BASE_PROJ = os.getcwd()
ARTIFACT_DIR = os.path.join(BASE_PROJ, "data", "processed")
_SCORER_OUT = os.path.join(BASE_PROJ, "models", "model_b", "neural")

sys.path.insert(0, _THIS_DIR)
os.makedirs(_SCORER_OUT, exist_ok=True)

# Import text utilities from model_a_train (which defines stubs)
# from src.model_a_train import word_tokens, sentence_fragments, normalize

_TFIDF_PATH = os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.pkl")
try:
    with open(_TFIDF_PATH, "rb") as _f:
        _TFIDF = pickle.load(_f)
except (FileNotFoundError, EOFError):
    _TFIDF = None

# ---------------------------------------------------------------------------
# Similarity helpers
# ---------------------------------------------------------------------------

def _batch_row_cosine(mat_a, mat_b) -> np.ndarray:
    """Row-wise cosine similarity between two same-shape sparse matrices."""
    dot    = np.array(mat_a.multiply(mat_b).sum(axis=1)).flatten()
    norm_a = np.sqrt(np.array(mat_a.power(2).sum(axis=1)).flatten())
    norm_b = np.sqrt(np.array(mat_b.power(2).sum(axis=1)).flatten())
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(norm_a * norm_b > 0, dot / (norm_a * norm_b), 0.0)

def _batch_transform(*lists) -> list:
    """Transform multiple string lists in one tqdm loop; returns list of sparse matrices."""
    mats, names = [], ["cand/sent", "gold/question", "passage"]
    for i, lst in enumerate(tqdm(lists, desc="  transform", unit="matrix")):
        mats.append(_TFIDF.transform(lst))
    return mats

# ---------------------------------------------------------------------------
# Keyword / candidate helpers
# ---------------------------------------------------------------------------

_OPTIONS, _NEG_PER_ITEM, _POOL_SIZE = ["A", "B", "C", "D"], 3, 15

def top_keywords(passage: str, k: int = 10) -> List[str]:
    viable = [t for t in word_tokens(passage) if len(t) >= 4]
    return [w for w, _ in Counter(viable).most_common(k)]

def _candidate_pool(passage: str) -> List[str]:
    return [w for w in top_keywords(passage, k=_POOL_SIZE) if w]

def _phrase_relevance(phrase: str, passage: str, gold: str) -> float:
    return passage.lower().count(phrase.lower()) * 0.7 + len(set(phrase.split()) & set(gold.split())) * 0.3

# ---------------------------------------------------------------------------
# Feature vectors  (pre-computed similarities passed in as scalars)
# ---------------------------------------------------------------------------

def _dist_feats(cand: str, gold: str, passage: str, sim_cg: float, sim_cp: float) -> List[float]:
    n = len(word_tokens(passage))
    return [
        sim_cg,
        sim_cp,
        len(set(cand) & set(gold)) / (len(gold) + 1e-9),
        passage.lower().count(cand.lower()) / (n + 1e-9),
        min(len(cand) / 20.0, 1.0),
        _phrase_relevance(cand, passage, gold),
    ]

def _hint_feats(sent: str, question: str, idx: int, total: int, sim: float) -> List[float]:
    q_toks, s_toks = set(word_tokens(question)), set(word_tokens(sent))
    return [len(q_toks & s_toks) / (len(q_toks) + 1e-9), idx / max(total - 1, 1), len(s_toks) / 50.0, sim]

# ---------------------------------------------------------------------------
# Dataset builders  — BATCHED
# ---------------------------------------------------------------------------

def _build_distractor_rows(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    cands, golds, passages, labels = [], [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  distractor collect", unit="row"):
        passage   = str(row["article"])
        gold_key  = str(row["answer"])
        gold_text = str(row[gold_key])
        wrong     = [str(row[o]) for o in _OPTIONS if o != gold_key]

        for opt in wrong:
            cands.append(opt); golds.append(gold_text); passages.append(passage); labels.append(1)

        kws = [kw for kw in _candidate_pool(passage) if 3 <= len(kw) <= 15 and kw not in wrong]
        for kw in kws[:_NEG_PER_ITEM]:
            cands.append(kw); golds.append(gold_text); passages.append(passage); labels.append(0)

    mat_c, mat_g, mat_p = _batch_transform(cands, golds, passages)
    sim_cg = _batch_row_cosine(mat_c, mat_g)
    sim_cp = _batch_row_cosine(mat_c, mat_p)

    X = np.array(
        [_dist_feats(cands[i], golds[i], passages[i], sim_cg[i], sim_cp[i])
         for i in tqdm(range(len(cands)), desc="  distractor features", unit="ex")],
        dtype=np.float32,
    )
    return X, np.array(labels, dtype=np.int32)


def _build_hint_rows(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    sentences, questions, sent_indices, total_counts = [], [], [], []
    group_starts, group_sizes = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  hint collect", unit="row"):
        sents = sentence_fragments(str(row["article"]))
        if not sents:
            continue
        q = str(row["question"])
        start = len(sentences)
        for idx, s in enumerate(sents):
            sentences.append(s); questions.append(q)
            sent_indices.append(idx); total_counts.append(len(sents))
        group_starts.append(start); group_sizes.append(len(sents))

    mat_s, mat_q = _batch_transform(sentences, questions)
    sims = _batch_row_cosine(mat_s, mat_q)

    labels = np.zeros(len(sentences), dtype=np.int32)
    for start, size in zip(group_starts, group_sizes):
        labels[start + int(np.argmax(sims[start:start + size]))] = 1

    X = np.array(
        [_hint_feats(sentences[i], questions[i], sent_indices[i], total_counts[i], float(sims[i]))
         for i in tqdm(range(len(sentences)), desc="  hint features", unit="ex")],
        dtype=np.float32,
    )
    return X, labels

# ---------------------------------------------------------------------------
# Training & evaluation
# ---------------------------------------------------------------------------

def _fit_scorer(model_class, input_dim, X, y, **kw):
    """Train a small NN on handcrafted features."""
    pos = y.sum()
    neg = len(y) - pos
    pos_weight = neg / max(pos, 1)
    model = model_class(input_dim=input_dim)
    model = train_nn(
        model, X, y,
        epochs=kw.get("epochs", 20),
        batch_size=kw.get("batch_size", 256),
        lr=kw.get("lr", 1e-3),
        pos_weight=pos_weight,
        patience=kw.get("patience", 5),
        verbose=True,
    )
    return model


_fit_distractor = lambda X, y: _fit_scorer(DistractorScorer, X.shape[1], X, y)
_fit_hint       = lambda X, y: _fit_scorer(HintScorer,       X.shape[1], X, y)

fit_distractor_scorer = _fit_distractor
fit_hint_scorer       = _fit_hint


def _eval(clf, X, y) -> dict:
    p = clf.predict(X)
    return {
        "accuracy":  accuracy_score(y, p),
        "f1":        f1_score(y, p, zero_division=0),
        "precision": precision_score(y, p, zero_division=0),
        "recall":    recall_score(y, p, zero_division=0),
    }

# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------

_DEFAULT_N     = 3
_FALLBACK_HINTS = ["Look for the main detail", "Scan for key terms", "Use the passage context"]


def pick_distractors(passage: str, correct_answer: str, scorer: LogisticRegression, n: int = _DEFAULT_N) -> List[str]:
    pool = _candidate_pool(passage)
    if not pool:
        return []
    m = len(pool)
    mat_p, mat_g, mat_a = _batch_transform(pool, [correct_answer] * m, [passage] * m)
    X = np.array([_dist_feats(pool[i], correct_answer, passage, float(_batch_row_cosine(mat_p, mat_g)[i]),
                               float(_batch_row_cosine(mat_p, mat_a)[i])) for i in range(m)], dtype=np.float32)
    ranked = sorted(zip(scorer.predict_proba(X)[:, 1], pool), reverse=True)
    selected, seen = [], set()
    for _, cand in ranked:
        if cand[:4] not in seen:
            seen.add(cand[:4]); selected.append(cand)
        if len(selected) >= n:
            break
    return selected


def pick_hints(passage: str, question: str, scorer: LogisticRegression, n: int = _DEFAULT_N) -> List[str]:
    sents = sentence_fragments(passage)
    if not sents:
        return _FALLBACK_HINTS[:]
    mat_s, mat_q = _batch_transform(sents, [question] * len(sents))
    sims = _batch_row_cosine(mat_s, mat_q)
    X = np.array([_hint_feats(sents[i], question, i, len(sents), float(sims[i])) for i in range(len(sents))], dtype=np.float32)
    top = [s for _, s in sorted(zip(scorer.predict_proba(X)[:, 1], sents), reverse=True)[:n]]
    hints = []
    if top:       hints.append(f"Hint 1: Think about {' '.join(top_keywords(top[-1], k=4))}")
    if len(top)>1: hints.append(f"Hint 2: {top[1][:120]}")
    if len(top)>2: hints.append(f"Hint 3: {top[0][:150]}")
    return hints

# ---------------------------------------------------------------------------
# Pipeline
# ---------------------------------------------------------------------------

def run():
    SEP = "=" * 42
    print(f"[scorer_b] starting\n{SEP}")

    # Local paths (commented)
    # train_df = pd.read_csv("../data/processed/train_clean.csv")
    # val_df   = pd.read_csv("../data/processed/val_clean.csv")
    
    train_df = pd.read_csv(os.path.join(ARTIFACT_DIR, "train_clean.csv"))
    val_df   = pd.read_csv(os.path.join(ARTIFACT_DIR, "val_clean.csv"))
    print(f"train: {len(train_df):,}  |  val: {len(val_df):,}")

    print("\n--- distractor ---")
    X_d, y_d = _build_distractor_rows(train_df)
    print(f"  matrix {X_d.shape}  positives: {y_d.sum():,}")

    print("\n--- hint ---")
    X_h, y_h = _build_hint_rows(train_df)
    print(f"  matrix {X_h.shape}  positives: {y_h.sum():,}")

    print("\n--- fit ---")
    dist_clf = _fit_distractor(X_d, y_d)
    hint_clf = _fit_hint(X_h, y_h)

    # eval with metrics collection
    metrics_rows = []
    for split, build in [("train", lambda: (X_d, y_d, X_h, y_h)),
                          ("val",   lambda: (*_build_distractor_rows(val_df), *_build_hint_rows(val_df)))]:
        Xd, yd, Xh, yh = build()
        for name, clf, X, y in [("distractor", dist_clf, Xd, yd), ("hint", hint_clf, Xh, yh)]:
            m = _eval(clf, X, y)
            metrics_rows.append({
                "model": name,
                "split": split,
                "accuracy": m["accuracy"],
                "f1": m["f1"],
                "precision": m["precision"],
                "recall": m["recall"]
            })
            print(f"  [{name} {split}]  " + "  ".join(f"{k}={v:.4f}" for k, v in m.items()))

    save_checkpoint(dist_clf, os.path.join(_SCORER_OUT, "distractor.pt"))
    save_checkpoint(hint_clf,  os.path.join(_SCORER_OUT, "hint.pt"))

    # Export metrics to CSV
    _export_metrics_to_csv(metrics_rows)

    # Train transformer question generator + distractor generator
    print("\n--- transformer models ---")
    _train_transformer_qg_dg(train_df)

    print(f"\n[scorer_b] done — saved to {_SCORER_OUT}\n{SEP}")


def _export_metrics_to_csv(metrics_rows: list):
    """Export Model-B metrics to CSV."""
    import os
    # Local path (commented)
    # reports_dir = os.path.normpath(os.path.join(_THIS_DIR, "..", "data", "processed", "reports"))
    
    reports_dir = os.path.join(ARTIFACT_DIR, "reports")
    os.makedirs(reports_dir, exist_ok=True)
    
    df = pd.DataFrame(metrics_rows)
    csv_path = os.path.join(reports_dir, "model_b_metrics.csv")
    df.to_csv(csv_path, index=False)
    print(f"    → {csv_path}")


def _train_transformer_qg_dg(train_df):
    """Optional transformer QG + DG training if transformers is installed."""
    try:
        from src.transformer_train import train_question_generator, train_distractor_generator
    except ImportError:
        print("  transformers not installed — skipping")
        return
    try:
        print("  training question generator (FLAN-T5) …")
        train_question_generator(train_df, epochs=10, use_lora=True)
    except Exception as e:
        print(f"  QG failed: {e}")
    try:
        print("  training distractor generator (BART) …")
        train_distractor_generator(train_df, epochs=10, use_lora=True)
    except Exception as e:
        print(f"  DG failed: {e}")


if __name__ == "__main__":
    run()

## TRANSFORMER TRAIN

In [1]:
"""
transformer_train.py — Training pipelines for transformer-based MCQ models
Optimized for dual T4 GPUs on Kaggle with maximum throughput.

Three sub-pipelines:
  1. TransformerAnswerVerifier  (BERT)   — score P(correct) for (article, question, option)
  2. QuestionGenerator          (T5)     — generate question from (context, answer)
  3. DistractorGenerator        (BART)   — generate distractors from (question, correct)
"""

from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# from src.nn_models import (
#     DEVICE,
#     DistractorGenerator,
#     QuestionGenerator,
#     TransformerAnswerVerifier,
#     train_seq2seq,
#     train_transformer,
# )

try:
    _THIS_DIR = Path(__file__).resolve().parent
    _BASE = _THIS_DIR.parent
except NameError:
    _BASE = Path(os.getcwd())
_DATA_RAW = _BASE / "data" / "raw"
_DATA_PROC = _BASE / "data" / "processed"
_MODEL_A_DIR = _BASE / "models" / "model_a" / "transformer"
_MODEL_B_DIR = _BASE / "models" / "model_b" / "transformer"
os.makedirs(_MODEL_A_DIR, exist_ok=True)
os.makedirs(_MODEL_B_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# GPU / hardware setup
# ---------------------------------------------------------------------------

def _setup_gpus() -> Tuple[torch.device, int]:
    """
    Detect available GPUs and return (primary_device, gpu_count).
    Prints a summary so Kaggle logs show what hardware is being used.
    """
    n_gpus = torch.cuda.device_count()
    if n_gpus == 0:
        print("[GPU] No CUDA devices found — running on CPU.")
        return torch.device("cpu"), 0

    print(f"[GPU] {n_gpus} CUDA device(s) detected:")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f"       GPU {i}: {props.name}  |  {props.total_memory // 1024**3} GB VRAM")

    # Pin computation to GPU 0; DataParallel scatters across all visible GPUs.
    device = torch.device("cuda:0")

    # cuDNN autotuner — big win for fixed input sizes (BERT, T5, BART encoders).
    torch.backends.cudnn.benchmark = True
    # TF32 on Ampere gives ~3× matmul throughput with negligible accuracy loss.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    return device, n_gpus


DEVICE, N_GPUS = _setup_gpus()


def _wrap_dataparallel(model: nn.Module) -> nn.Module:
    """
    Wrap model with DataParallel when multiple GPUs are available.
    DataParallel is simpler than DDP for a single-process Kaggle notebook
    and requires zero changes to the training loop.
    """
    if N_GPUS > 1:
        print(f"[GPU] Wrapping model with DataParallel across {N_GPUS} GPUs.")
        model = nn.DataParallel(model)
    return model.to(DEVICE)


def _try_compile(model: nn.Module) -> nn.Module:
    """
    Attempt torch.compile (PyTorch ≥ 2.0).  Falls back gracefully on older
    versions or unsupported backends (e.g. Windows / CPU-only builds).
    """
    if hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
            print("[Speed] torch.compile() applied (reduce-overhead mode).")
        except Exception as e:
            print(f"[Speed] torch.compile() skipped: {e}")
    return model


# ---------------------------------------------------------------------------
# DataLoader monkey-patch — injects T4-optimal flags without touching nn_models
# ---------------------------------------------------------------------------

def _patch_dataloader() -> None:
    """
    Monkey-patch torch.utils.data.DataLoader so that every DataLoader
    constructed anywhere in this process (including inside nn_models.py)
    automatically gets pin_memory, num_workers, prefetch_factor, and
    persistent_workers — without requiring any signature changes in the
    underlying training helpers.

    Safe to call multiple times (idempotent).
    """
    import torch.utils.data as _tud

    if getattr(_tud.DataLoader, "_t4_patched", False):
        return  # already patched

    if DEVICE.type == "cpu":
        print("[Speed] CPU-only mode — DataLoader patch skipped.")
        return

    _OrigDataLoader = _tud.DataLoader
    n_workers = min(4 * max(N_GPUS, 1), 8)  # cap at 8; Kaggle gives 4 vCPUs

    class _T4DataLoader(_OrigDataLoader):
        """DataLoader subclass with T4-optimal defaults baked in."""
        _t4_patched = True

        def __init__(self, *args, **kwargs):
            # Only inject our defaults when the caller didn't set them.
            if "num_workers" not in kwargs:
                kwargs["num_workers"] = n_workers
            if "pin_memory" not in kwargs:
                kwargs["pin_memory"] = True
            # prefetch_factor and persistent_workers require num_workers > 0.
            if kwargs.get("num_workers", 0) > 0:
                if "prefetch_factor" not in kwargs:
                    kwargs["prefetch_factor"] = 2
                if "persistent_workers" not in kwargs:
                    kwargs["persistent_workers"] = True
            super().__init__(*args, **kwargs)

    _tud.DataLoader = _T4DataLoader
    # Also patch the symbol that callers may have already imported directly:
    # e.g. `from torch.utils.data import DataLoader` in nn_models.py.
    # We can't retroactively patch those imports, but patching the module
    # attribute covers any code that goes through `torch.utils.data.DataLoader`.
    print(f"[Speed] DataLoader patched: num_workers={n_workers}, "
          "pin_memory=True, prefetch_factor=2, persistent_workers=True.")


_patch_dataloader()  # Apply immediately at import time.


# ---------------------------------------------------------------------------
# Data preparation helpers
# ---------------------------------------------------------------------------

def _load_csv(path) -> pd.DataFrame:
    if not path.exists():
        print(f"  WARNING: {path} not found")
        return pd.DataFrame()
    df = pd.read_csv(path, low_memory=False)
    required = {"article", "question", "A", "B", "C", "D", "answer"}
    if not required.issubset(df.columns):
        print(f"  WARNING: {path} missing columns. Got {list(df.columns)}")
        return pd.DataFrame()
    return df


def prepare_answer_verifier_data(
    df: pd.DataFrame,
    max_rows: Optional[int] = None,
) -> Tuple[List[str], np.ndarray]:
    """
    Convert a MCQ DataFrame into (texts, labels) for TransformerAnswerVerifier.
    Each row → 4 examples, one per option.  Label = 1 for correct option.
    """
    texts, labels = [], []
    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=42)
    for _, row in df.iterrows():
        correct_letter = str(row["answer"]).strip().upper()
        for opt_letter in ["A", "B", "C", "D"]:
            option_text = str(row.get(opt_letter, ""))
            text = (
                f"article: {str(row['article'])}\n"
                f"question: {str(row['question'])}\n"
                f"option: {option_text}"
            )
            texts.append(text)
            labels.append(1 if opt_letter == correct_letter else 0)
    return texts, np.array(labels, dtype=np.int32)


def prepare_question_generation_data(
    df: pd.DataFrame,
    max_rows: Optional[int] = None,
) -> Tuple[List[str], List[str]]:
    """
    Convert a MCQ DataFrame into (inputs, targets) for QuestionGenerator.
    Input:  "context: {article} answer: {correct_option}"
    Target: the question text
    """
    inputs, targets = [], []
    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=42)
    for _, row in df.iterrows():
        correct_letter = str(row["answer"]).strip().upper()
        correct_text = str(row.get(correct_letter, ""))
        article = str(row["article"])
        question = str(row.get("question", ""))
        if not question or len(question) < 5:
            continue
        inputs.append(f"context: {article} answer: {correct_text}")
        targets.append(question)
    return inputs, targets


def prepare_distractor_data(
    df: pd.DataFrame,
    max_rows: Optional[int] = None,
) -> Tuple[List[str], List[str]]:
    """
    Convert MCQ DataFrame into (inputs, targets) for DistractorGenerator.
    Input:  "question: {question} correct: {correct_option}"
    Target: "distractor1 | distractor2 | distractor3"
    """
    inputs, targets = [], []
    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=42)
    for _, row in df.iterrows():
        correct_letter = str(row["answer"]).strip().upper()
        correct_text = str(row.get(correct_letter, ""))
        question = str(row.get("question", ""))
        if not question or len(question) < 5:
            continue
        distractors = [
            str(row.get(opt, ""))
            for opt in ["A", "B", "C", "D"]
            if opt != correct_letter
        ]
        distractors = [d for d in distractors if d and d != correct_text]
        if len(distractors) < 3:
            continue
        inputs.append(f"question: {question} correct: {correct_text}")
        targets.append(" | ".join(distractors[:3]))
    return inputs, targets


# ---------------------------------------------------------------------------
# Training wrappers
# ---------------------------------------------------------------------------

def train_answer_verifier_transformer(
    train_df: pd.DataFrame,
    val_df: Optional[pd.DataFrame] = None,
    model_name: str = "bert-base-uncased",
    epochs: int = 3,
    # Default batch size scaled up for dual-GPU (2× 16 GB VRAM on T4).
    # DataParallel splits each batch across both cards, so effective per-GPU
    # batch = batch_size / N_GPUS.
    batch_size: int = 32,
    lr: float = 2e-5,
    use_lora: bool = True,
    lora_r: int = 8,
    max_length: int = 256,
    fp16: bool = True,   # AMP handled internally via GradScaler
    max_rows: Optional[int] = None,
    checkpoint_dir: Optional[Path] = None,
    resume_from_checkpoint: Optional[Path] = None,
):
    """Train a BERT-based answer verifier on MCQ data (dual-GPU optimized)."""
    print("[TransformerAV] preparing data …")
    train_texts, train_labels = prepare_answer_verifier_data(train_df, max_rows)

    val_texts, val_labels = None, None
    if val_df is not None and len(val_df) > 0:
        val_texts, val_labels = prepare_answer_verifier_data(val_df, max_rows)

    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print(f"[TransformerAV] training {model_name}  |  train={len(train_texts):,}  val={len(val_texts) if val_texts else 0:,}")
    model = TransformerAnswerVerifier(
        model_name=model_name,
        num_labels=2,
        use_lora=use_lora,
        lora_r=lora_r,
    )

    # Multi-GPU + compile
    model = _try_compile(model)
    model = _wrap_dataparallel(model)

    print(f"[TransformerAV] tokenizing {len(train_texts):,} train"
          f" + {len(val_texts) if val_texts else 0:,} val texts "
          f"(max_length={max_length}) …")
    # fp16/AMP is handled automatically inside train_transformer via GradScaler.
    # DataLoader speed flags (pin_memory, num_workers, prefetch) are injected
    # globally via the DataLoader monkey-patch in _patch_dataloader() above.
    model = train_transformer(
        model=model,
        tokenizer=tokenizer,
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=val_texts,
        val_labels=val_labels,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=2,
        max_length=max_length,
        checkpoint_dir=checkpoint_dir or (_MODEL_A_DIR / "checkpoints"),
        resume_from_checkpoint=resume_from_checkpoint,
    )

    # Save — unwrap DataParallel before persisting state_dict so the checkpoint
    # is portable (loadable on a single-GPU or CPU machine without DataParallel).
    path = _MODEL_A_DIR / "answer_verifier_transformer.pt"
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state, path)
    with open(_MODEL_A_DIR / "transformer_meta.json", "w") as f:
        json.dump({"model_name": model_name, "type": "answer_verifier"}, f)
    print(f"[TransformerAV] saved → {path}")
    return model


def train_question_generator(
    df: pd.DataFrame,
    model_name: str = "google/flan-t5-base",
    epochs: int = 5,
    # Seq2seq models are decoder-heavy; keep per-GPU batch modest.
    batch_size: int = 16,
    lr: float = 3e-5,
    use_lora: bool = True,
    lora_r: int = 8,
    fp16: bool = True,   # AMP handled internally via GradScaler
    max_rows: Optional[int] = None,
    checkpoint_dir: Optional[Path] = None,
    resume_from_checkpoint: Optional[Path] = None,
):
    """Train a T5/FLAN-T5 question generator (dual-GPU optimized)."""
    print("[QG] preparing data …")
    inputs, targets = prepare_question_generation_data(df, max_rows)
    split = int(len(inputs) * 0.9)
    tr_in, tr_tg = inputs[:split], targets[:split]
    val_in, val_tg = (inputs[split:], targets[split:]) if split < len(inputs) else (None, None)

    print(f"[QG] training {model_name}  |  train={len(tr_in):,}  val={len(val_in) if val_in else 0:,}")
    model = QuestionGenerator(
        model_name=model_name,
        use_lora=use_lora,
        lora_r=lora_r,
    )

    model = _try_compile(model)
    model = _wrap_dataparallel(model)

    # fp16/AMP handled internally via GradScaler inside train_seq2seq.
    model = train_seq2seq(
        model=model,
        train_inputs=tr_in,
        train_targets=tr_tg,
        val_inputs=val_in,
        val_targets=val_tg,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=2,
        checkpoint_dir=checkpoint_dir or (_MODEL_B_DIR / "checkpoints"),
        resume_from_checkpoint=resume_from_checkpoint,
    )

    path = _MODEL_B_DIR / "question_generator.pt"
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state, path)
    with open(_MODEL_B_DIR / "qg_meta.json", "w") as f:
        json.dump({"model_name": model_name, "type": "question_generator"}, f)
    print(f"[QG] saved → {path}")
    return model


def train_distractor_generator(
    df: pd.DataFrame,
    model_name: str = "facebook/bart-base",
    epochs: int = 5,
    batch_size: int = 16,
    lr: float = 3e-5,
    use_lora: bool = True,
    lora_r: int = 8,
    fp16: bool = True,   # AMP handled internally via GradScaler
    max_rows: Optional[int] = None,
    checkpoint_dir: Optional[Path] = None,
    resume_from_checkpoint: Optional[Path] = None,
):
    """Train a BART/T5 distractor generator (dual-GPU optimized)."""
    print("[DG] preparing data …")
    inputs, targets = prepare_distractor_data(df, max_rows)
    split = int(len(inputs) * 0.9)
    tr_in, tr_tg = inputs[:split], targets[:split]
    val_in, val_tg = (inputs[split:], targets[split:]) if split < len(inputs) else (None, None)

    print(f"[DG] training {model_name}  |  train={len(tr_in):,}  val={len(val_in) if val_in else 0:,}")
    model = DistractorGenerator(
        model_name=model_name,
        use_lora=use_lora,
        lora_r=lora_r,
    )

    model = _try_compile(model)
    model = _wrap_dataparallel(model)

    model = train_seq2seq(
        model=model,
        train_inputs=tr_in,
        train_targets=tr_tg,
        val_inputs=val_in,
        val_targets=val_tg,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        patience=2,
        checkpoint_dir=checkpoint_dir or (_MODEL_B_DIR / "checkpoints"),
        resume_from_checkpoint=resume_from_checkpoint,
    )

    path = _MODEL_B_DIR / "distractor_generator.pt"
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(state, path)
    with open(_MODEL_B_DIR / "dg_meta.json", "w") as f:
        json.dump({"model_name": model_name, "type": "distractor_generator"}, f)
    print(f"[DG] saved → {path}")
    return model


# ---------------------------------------------------------------------------
# Full pipeline
# ---------------------------------------------------------------------------

def run_all(
    csv_path: Optional[str] = None,
    model_a_model: str = "bert-base-uncased",
    qg_model: str = "google/flan-t5-base",
    dg_model: str = "facebook/bart-base",
    max_rows: Optional[int] = None,
    epochs_av: int = 3,
    epochs_qg: int = 5,
    epochs_dg: int = 5,
    fp16: bool = True,
    av_resume_from: Optional[str] = None,
    qg_resume_from: Optional[str] = None,
    dg_resume_from: Optional[str] = None,
):
    """End-to-end training of all three transformer models."""
    # Load data
    path = Path(csv_path) if csv_path else _DATA_RAW / "train.csv"
    print(f"Loading data from {path}")
    df = _load_csv(path)
    if len(df) == 0:
        print("No data loaded. Aborting.")
        return

    split = int(len(df) * 0.85)
    train_df = df.iloc[:split]
    val_df = df.iloc[split:]
    print(f"Data: {len(df):,} rows  |  train={len(train_df):,}  val={len(val_df):,}")

    """1 is done, have the file, skipping this section"""
    # print("\n" + "=" * 60)
    # print("1. Training Transformer AnswerVerifier")
    # print("=" * 60)
    # train_answer_verifier_transformer(
    #     train_df, val_df,
    #     model_name=model_a_model,
    #     epochs=epochs_av,
    #     fp16=fp16,
    #     max_rows=max_rows,
    #     resume_from_checkpoint=Path(av_resume_from) if av_resume_from else None,
    # )

    print("\n" + "=" * 60)
    print("2. Training Question Generator (T5/FLAN-T5)")
    print("=" * 60)
    train_question_generator(
        train_df,
        model_name=qg_model,
        epochs=epochs_qg,
        fp16=fp16,
        max_rows=max_rows,
        resume_from_checkpoint=Path(qg_resume_from) if qg_resume_from else None,
    )

    print("\n" + "=" * 60)
    print("3. Training Distractor Generator (BART)")
    print("=" * 60)
    train_distractor_generator(
        train_df,
        model_name=dg_model,
        epochs=epochs_dg,
        fp16=fp16,
        max_rows=max_rows,
        resume_from_checkpoint=Path(dg_resume_from) if dg_resume_from else None,
    )

    print("\n" + "=" * 60)
    print("Done. Models saved to:")
    print(f"  {_MODEL_A_DIR}")
    print(f"  {_MODEL_B_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Train transformer models for MCQ pipeline")
    parser.add_argument("--csv", default=None, help="Path to training CSV")
    parser.add_argument("--av-model", default="bert-base-uncased")
    parser.add_argument("--qg-model", default="google/flan-t5-base")
    parser.add_argument("--dg-model", default="facebook/bart-base")
    parser.add_argument("--max-rows", type=int, default=None)
    parser.add_argument("--epochs-av", type=int, default=3)
    parser.add_argument("--epochs-qg", type=int, default=5)
    parser.add_argument("--epochs-dg", type=int, default=5)
    parser.add_argument("--fp16", dest="fp16", action="store_true", default=True,
                        help="Use mixed-precision training (default on)")
    parser.add_argument("--no-fp16", dest="fp16", action="store_false",
                        help="Disable mixed-precision training")
    parser.add_argument("--av-resume-from", default=None,
                        help="Path to a transformer checkpoint to resume answer-verifier training")
    parser.add_argument("--qg-resume-from", default=None,
                        help="Path to a transformer checkpoint to resume question-generator training")
    parser.add_argument("--dg-resume-from", default=None,
                        help="Path to a transformer checkpoint to resume distractor-generator training")
    args, _ = parser.parse_known_args()
    run_all(
        csv_path=args.csv,
        model_a_model=args.av_model,
        qg_model=args.qg_model,
        dg_model=args.dg_model,
        max_rows=args.max_rows,
        epochs_av=args.epochs_av,
        epochs_qg=args.epochs_qg,
        epochs_dg=args.epochs_dg,
        fp16=args.fp16,
        av_resume_from=args.av_resume_from,
        qg_resume_from=args.qg_resume_from,
        dg_resume_from=args.dg_resume_from,
    )

[GPU] 2 CUDA device(s) detected:
       GPU 0: Tesla T4  |  14 GB VRAM
       GPU 1: Tesla T4  |  14 GB VRAM
[Speed] DataLoader patched: num_workers=8, pin_memory=True, prefetch_factor=2, persistent_workers=True.
Loading data from /kaggle/working/data/raw/train.csv
Data: 279,035 rows  |  train=237,179  val=41,856

2. Training Question Generator (T5/FLAN-T5)
[QG] preparing data …
[QG] training google/flan-t5-base  |  train=113,528  val=12,615


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561
[GPU] Wrapping model with DataParallel across 2 GPUs.
[Speed] torch.compile() applied (reduce-overhead mode).


AttributeError: 'DataParallel' object has no attribute 'tokenizer'

In [3]:
!nvidia-smi


Thu Jun 11 11:50:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             31W /   70W |    2719MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## INFERENCE

In [ ]:
"""
inference.py
Unified inference API — used by the Streamlit UI and the Colab notebook.

Public functions:
  run_inference(article, race_rows=None) → {'questions': [...], 'latency_ms': N}
  verify_answer(article, question, chosen_text, correct_text) → dict
  get_model_metrics() → dict
"""

import os
import sys
import pickle
import re
import time
import random

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity as _sk_cos
from scipy.sparse import hstack as sparse_hstack

try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    PROJECT_ROOT = os.path.abspath(os.path.join(_THIS_DIR, '..'))
except NameError:
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import clean_text, tokenize, split_into_sentences, tfidf_cosine, PROCESSED_DIR, BASE_DIR
from src.model_a_train import generate_questions_from_passage, make_sample_string, verify_answer as verify_answer_ma
from src.nn_models import load_checkpoint, AnswerVerifier, DistractorScorer, HintScorer, TransformerAnswerVerifier

try:
    from model_b_train import generate_distractors, generate_hints
except Exception:
    def generate_distractors(article, answer, model, n=3):
        return []

    def generate_hints(article, question, model, n=3):
        sents = split_into_sentences(article)
        if not sents:
            return ["Read carefully", "Focus keywords", "Check passage"]
        return [f"Hint {i+1}: {s[:100]} …" for i, s in enumerate(sents[:n])]

MODEL_A_DIR = os.path.join(BASE_DIR, 'models', 'model_a', 'neural')
MODEL_A_TRANSFORMER_DIR = os.path.join(BASE_DIR, 'models', 'model_a', 'transformer')
MODEL_B_DIR = os.path.join(BASE_DIR, 'models', 'model_b', 'neural')
MODEL_B_TRANSFORMER_DIR = os.path.join(BASE_DIR, 'models', 'model_b', 'transformer')

MIN_QUIZ_QUESTIONS = 5
MAX_QUIZ_QUESTIONS = 10


_cache = {}


def _try_load_transformers(cache):
    """Load transformer models if available."""
    try:
        from transformers import AutoTokenizer
    except ImportError:
        return

    # Transformer answer verifier
    meta_path = os.path.join(MODEL_A_TRANSFORMER_DIR, 'transformer_meta.json')
    ckpt_path = os.path.join(MODEL_A_TRANSFORMER_DIR, 'answer_verifier_transformer.pt')
    if os.path.exists(ckpt_path) and os.path.exists(meta_path):
        try:
            import json
            with open(meta_path) as f:
                meta = json.load(f)
            tokenizer = AutoTokenizer.from_pretrained(meta.get('model_name', 'bert-base-uncased'))
            model = TransformerAnswerVerifier(model_name=meta['model_name'], num_labels=2)
            model.load_state_dict(torch.load(ckpt_path, map_location='cpu', weights_only=True))
            model.eval()
            cache['transformer_model'] = model
            cache['transformer_tokenizer'] = tokenizer
            print(f"  loaded transformer answer verifier ({meta['model_name']})")
        except Exception as e:
            print(f"  WARNING: transformer model load failed: {e}")

    # Question generator
    qg_path = os.path.join(MODEL_B_TRANSFORMER_DIR, 'question_generator.pt')
    qg_meta_path = os.path.join(MODEL_B_TRANSFORMER_DIR, 'qg_meta.json')
    if os.path.exists(qg_path) and os.path.exists(qg_meta_path):
        try:
            import json
            with open(qg_meta_path) as f:
                meta = json.load(f)
            from src.nn_models import QuestionGenerator
            cache['question_generator'] = QuestionGenerator(
                model_name=meta['model_name'], use_lora=False
            )
            cache['question_generator'].load_state_dict(
                torch.load(qg_path, map_location='cpu', weights_only=True)
            )
            cache['question_generator'].eval()
            print(f"  loaded question generator ({meta['model_name']})")
        except Exception as e:
            print(f"  WARNING: question generator load failed: {e}")

    # Distractor generator
    dg_path = os.path.join(MODEL_B_TRANSFORMER_DIR, 'distractor_generator.pt')
    dg_meta_path = os.path.join(MODEL_B_TRANSFORMER_DIR, 'dg_meta.json')
    if os.path.exists(dg_path) and os.path.exists(dg_meta_path):
        try:
            import json
            with open(dg_meta_path) as f:
                meta = json.load(f)
            from src.nn_models import DistractorGenerator
            cache['distractor_generator'] = DistractorGenerator(
                model_name=meta['model_name'], use_lora=False
            )
            cache['distractor_generator'].load_state_dict(
                torch.load(dg_path, map_location='cpu', weights_only=True)
            )
            cache['distractor_generator'].eval()
            print(f"  loaded distractor generator ({meta['model_name']})")
        except Exception as e:
            print(f"  WARNING: distractor generator load failed: {e}")


def load_models():
    """Load all trained models from disk on first call; return cache thereafter."""
    if _cache:
        return _cache

    # Load NN answer verifier
    nn_path = os.path.join(MODEL_A_DIR, 'answer_verifier.pt')
    if os.path.exists(nn_path):
        try:
            ckpt = torch.load(nn_path, map_location='cpu', weights_only=True)
            meta = ckpt.get('meta', {})
            input_dim = meta.get('input_dim', 10000)
            model = AnswerVerifier(input_dim=input_dim)
            model.load_state_dict(ckpt['model_state_dict'])
            model.eval()
            _cache['model'] = model
        except Exception as e:
            print(f"  WARNING: could not load NN model: {e}")
            _cache['model'] = None
    else:
        print(f"  WARNING: NN checkpoint not found at {nn_path}.")
        _cache['model'] = None

    # Load NN distractor scorer
    dist_path = os.path.join(MODEL_B_DIR, 'distractor.pt')
    if os.path.exists(dist_path):
        try:
            _cache['dist_ranker'] = load_checkpoint(DistractorScorer, dist_path)
        except Exception as e:
            print(f"  WARNING: could not load distractor model: {e}")
            _cache['dist_ranker'] = None
    else:
        print(f"  WARNING: distractor checkpoint not found at {dist_path}.")
        _cache['dist_ranker'] = None

    # Load NN hint scorer
    hint_path = os.path.join(MODEL_B_DIR, 'hint.pt')
    if os.path.exists(hint_path):
        try:
            _cache['hint_scorer'] = load_checkpoint(HintScorer, hint_path)
        except Exception as e:
            print(f"  WARNING: could not load hint model: {e}")
            _cache['hint_scorer'] = None
    else:
        print(f"  WARNING: hint checkpoint not found at {hint_path}.")
        _cache['hint_scorer'] = None

    # Try loading transformer models
    _try_load_transformers(_cache)

    def _try_pickle(path, label):
        try:
            with open(path, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"  WARNING: could not load {label} ({path}): {e}")
            return None

    for pkl_key, fname in [('ohe', 'ohe_vectorizer.pkl'), ('tfidf', 'tfidf_vectorizer.pkl')]:
        path = os.path.join(PROCESSED_DIR, fname)
        if os.path.exists(path):
            _cache[pkl_key] = _try_pickle(path, fname)
        else:
            alt = os.path.join(PROCESSED_DIR, 'tfidf.pkl') if pkl_key == 'tfidf' else None
            if alt and os.path.exists(alt):
                _cache[pkl_key] = _try_pickle(alt, 'tfidf.pkl')
            else:
                print(f"  WARNING: {fname} not found. Run preprocessing / training first.")
                _cache[pkl_key] = None

    return _cache


def remove_answer_from_question(question_text, answer_text):
    """Utility: replace answer tokens in question with '___'."""
    import re
    for token in tokenize(answer_text):
        if len(token) > 3:
            pattern = r'\b' + re.escape(token) + r'\b'
            question_text = re.sub(pattern, '___', question_text, flags=re.IGNORECASE)
    return question_text


def shuffle_options(correct_answer, distractors):
    """Randomly assign correct answer + 3 distractors to slots A/B/C/D."""
    pool = [correct_answer] + distractors[:3]
    random.shuffle(pool)
    idx  = pool.index(correct_answer)
    lbls = ['A', 'B', 'C', 'D']
    return {lbls[i]: pool[i] for i in range(4)}, lbls[idx]


def _get_hints(article, question, models):
    if models.get('hint_scorer'):
        return generate_hints(article, question, models['hint_scorer'], n=3)
    sents = split_into_sentences(article)
    return [f"Hint {i+1}: {s[:100]} …" for i, s in enumerate(sents[:3])]


# ---------------------------------------------------------------------------
# Handcrafted features + ensemble prediction
# ---------------------------------------------------------------------------

ENSEMBLE_WEIGHTS = {
    "nn": 0.35,
    "transformer": 0.50,
    "tfidf": 0.15,
}


def _compute_hc_row(article, question, opt_text, tfidf_vec, idf_dict=None, vocab=None):
    """Compute 12 handcrafted features for one option (mirrors preprocessing.py)."""
    if tfidf_vec is not None and len(opt_text) > 0:
        art_vec = tfidf_vec.transform([article])
        q_vec = tfidf_vec.transform([question])
        opt_vec = tfidf_vec.transform([opt_text])
        sim_ao = float(_sk_cos(art_vec, opt_vec)[0, 0])
        sim_qo = float(_sk_cos(q_vec, opt_vec)[0, 0])
        opt_tfidf_max = float(opt_vec.max())
    else:
        sim_ao = sim_qo = 0.0
        opt_tfidf_max = 0.0

    art_tokens = set(article.split())
    opt_tokens = set(opt_text.split())
    q_tokens = set(question.split())
    overlap_ao = len(opt_tokens & art_tokens) / (len(opt_tokens) + 1e-9)
    overlap_qo = len(opt_tokens & q_tokens) / (len(opt_tokens) + 1e-9)
    opt_len_ratio = min(len(opt_text) / 200.0, 1.0)
    overlap_oa = len(opt_tokens & art_tokens) / (len(art_tokens) + 1e-9)
    overlap_oq = len(opt_tokens & q_tokens) / (len(q_tokens) + 1e-9)

    opt_idf_mean = 0.0
    if idf_dict and vocab and opt_tokens:
        idfs = [idf_dict.get(w, 0.0) for w in opt_tokens if w in vocab]
        opt_idf_mean = sum(idfs) / len(idfs) if idfs else 0.0

    has_digit = 1.0 if re.search(r"\d", opt_text) else 0.0
    art_bigrams = set(" ".join(article.split()[i:i+2]) for i in range(len(article.split()) - 1))
    opt_bigrams = set(" ".join(opt_text.split()[i:i+2]) for i in range(len(opt_text.split()) - 1))
    shared_bigrams = len(art_bigrams & opt_bigrams) / (len(opt_bigrams) + 1e-9) if opt_bigrams else 0.0
    starts_cap = 1.0 if opt_text and opt_text[0].isupper() else 0.0

    return [sim_ao, sim_qo, overlap_ao, overlap_qo, opt_len_ratio,
            overlap_oa, overlap_oq, opt_idf_mean, has_digit,
            shared_bigrams, opt_tfidf_max, starts_cap]


def _nn_predict_proba(article, question, option_texts, nn_model, ohe_vec, tfidf_vec):
    """P(correct) for each option from the NN with handcrafted features."""
    idf_dict = dict(zip(tfidf_vec.get_feature_names_out(), tfidf_vec.idf_)) if tfidf_vec is not None else None
    vocab = set(tfidf_vec.get_feature_names_out()) if tfidf_vec is not None else None
    samples = ohe_vec.transform([make_sample_string(article, question, t) for t in option_texts])
    if tfidf_vec is not None:
        hc = np.array([_compute_hc_row(article, question, t, tfidf_vec, idf_dict, vocab) for t in option_texts], dtype=np.float32)
        samples = sparse_hstack([samples, hc.astype(np.float32)], format="csr")
    return nn_model.predict_proba(samples)[:, 1]


def _transformer_predict_proba(article, question, option_texts, trf_model, tokenizer):
    """P(correct) for each option from the transformer."""
    from src.model_a_train import verify_answer_transformer
    probs = []
    for t in option_texts:
        out = verify_answer_transformer(article, question, t, trf_model, tokenizer)
        probs.append(out["probability"])
    return np.array(probs)


def _tfidf_scores(article, question, option_texts, tfidf_vec):
    """Normalized TF-IDF cosine similarity scores for each option."""
    scores = np.array([tfidf_cosine(article, t, tfidf_vec) for t in option_texts])
    total = scores.sum() + 1e-9
    return scores / total


def _build_fallback_distractors(article, correct_answer, n=3):
    ans_set = set(tokenize(correct_answer))
    pool = list(dict.fromkeys([
        t for t in tokenize(article) if t not in ans_set and len(t) > 3
    ]))
    while len(pool) < n:
        pool.append(f"option {len(pool) + 1}")
    return pool[:n]


def _model_a_pack(models):
    """Return a dict that verify_answer_ma (the NN-based one) expects."""
    return {'model': models.get('model')}


def _build_fallback_question(article, idx, models):
    """Create a safe fallback MCQ so we can always reach the minimum quiz size."""
    sentences = split_into_sentences(article)
    source = sentences[min(idx, max(len(sentences) - 1, 0))] if sentences else article
    source = source.strip() if source else article.strip()

    question_text = f"Which statement best matches the passage detail #{idx + 1}?"
    correct_answer = source[:120] if source else "A key point from the passage"
    distractors = _build_fallback_distractors(article, correct_answer, n=3)
    options, correct_label = shuffle_options(correct_answer, distractors)
    hints = _get_hints(article, question_text, models)

    return {
        'question': question_text,
        'correct_answer': correct_answer,
        'correct_label': correct_label,
        'options': options,
        'hints': hints,
        'source_sentence': source[:200],
    }


def build_question_from_race_row(article, row, models):
    correct_letter = str(row.get('answer', 'A')).strip().upper()
    correct_answer = str(row.get(correct_letter, ''))
    hints = _get_hints(article, str(row.get('question', '')), models)

    return {
        'question':        str(row.get('question', '')),
        'correct_answer':  correct_answer,
        'correct_label':   correct_letter,
        'options': {
            'A': str(row.get('A', '')),
            'B': str(row.get('B', '')),
            'C': str(row.get('C', '')),
            'D': str(row.get('D', '')),
        },
        'hints':           hints,
        'source_sentence': article[:200],
    }


def build_question_from_generated(article, item, models):
    """Map model_a_train.generate_questions_from_passage() dict to UI shape."""
    q_text = item['question']
    opts = item.get('options') or {}
    correct_letter = str(item.get('correct_letter', 'A')).strip().upper()
    correct_answer = item.get('answer', opts.get(correct_letter, ''))

    if models.get('dist_ranker') and correct_answer:
        distractors = generate_distractors(
            article, correct_answer, models['dist_ranker'], n=3
        )
        opts2, correct_letter = shuffle_options(correct_answer, distractors)
        opts = opts2

    hints = _get_hints(article, q_text, models)
    return {
        'question':        q_text,
        'correct_answer':  correct_answer,
        'correct_label':   correct_letter,
        'options':         opts,
        'hints':           hints,
        'source_sentence': str(item.get('source_sentence', '')),
    }


def run_inference(article, race_rows=None):
    t0 = time.time()
    models = load_models()

    question_list = []

    if race_rows:
        for row in race_rows[:MAX_QUIZ_QUESTIONS]:
            question_list.append(build_question_from_race_row(article, row, models))

    n_needed = MIN_QUIZ_QUESTIONS - len(question_list)
    if n_needed > 0:
        try:
            raw = generate_questions_from_passage(article, count=n_needed + 2)
            for item in raw[:n_needed]:
                question_list.append(build_question_from_generated(article, item, models))
        except Exception as e:
            pass

    # Guarantee minimum count even when model generation returns too few items.
    while len(question_list) < MIN_QUIZ_QUESTIONS:
        question_list.append(_build_fallback_question(article, len(question_list), models))

    # Never exceed the configured maximum quiz size.
    question_list = question_list[:MAX_QUIZ_QUESTIONS]

    latency_ms = int((time.time() - t0) * 1000)
    print(f"  run_inference: {len(question_list)} questions in {latency_ms} ms")

    return {'questions': question_list, 'latency_ms': latency_ms}


def verify_answer(article, question, chosen_option_text, correct_answer_text):
    models = load_models()
    is_correct = clean_text(chosen_option_text) == clean_text(correct_answer_text)

    def _norm_prob(p_chosen, p_correct):
        return p_chosen / (p_chosen + p_correct + 1e-9)

    # --- Ensemble: combine NN + transformer + TF-IDF ---
    option_texts = [chosen_option_text, correct_answer_text]
    weights = {}
    probs = {}

    # NN
    if models.get('model') and models.get('ohe') and models.get('tfidf'):
        try:
            p = _nn_predict_proba(
                clean_text(article), clean_text(question), option_texts,
                models['model'], models['ohe'], models['tfidf'],
            )
            probs['nn'] = p
            weights['nn'] = ENSEMBLE_WEIGHTS['nn']
        except Exception:
            pass

    # Transformer
    if models.get('transformer_model') and models.get('transformer_tokenizer'):
        try:
            p = _transformer_predict_proba(
                article, question, option_texts,
                models['transformer_model'], models['transformer_tokenizer'],
            )
            probs['transformer'] = p
            weights['transformer'] = ENSEMBLE_WEIGHTS['transformer']
        except Exception:
            pass

    # TF-IDF
    if models.get('tfidf'):
        try:
            probs['tfidf'] = _tfidf_scores(article, question, option_texts, models['tfidf'])
            weights['tfidf'] = ENSEMBLE_WEIGHTS['tfidf']
        except Exception:
            pass

    if weights:
        total_w = sum(weights.values())
        p_ens = sum(weights[k] * probs[k] for k in weights) / total_w
        conf = p_ens[0]  # P(correct) for chosen_text
        method = "ensemble (" + "+".join(weights.keys()) + ")"
        return {
            'is_correct': is_correct,
            'confidence': conf if is_correct else (1.0 - conf),
            'method': method,
        }

    # --- Fallback: direct text match ---
    return {
        'is_correct': is_correct,
        'confidence': 1.0 if is_correct else 0.0,
        'method': 'direct match',
    }


def get_model_metrics():
    reports_dir = os.path.join(PROCESSED_DIR, 'reports')
    metrics = {}

    ma_csv = os.path.join(reports_dir, 'model_a_binary_metrics.csv')
    if os.path.exists(ma_csv):
        df = pd.read_csv(ma_csv)
        ma = {}
        for _, row in df.iterrows():
            model = row.get('model', 'nn')
            ma[model.lower()] = {
                'accuracy':  float(row.get('accuracy', 0)),
                'precision': float(row.get('precision', 0)),
                'recall':    float(row.get('recall', 0)),
                'f1':        float(row.get('f1', 0)),
                '4way_acc':  float(row.get('4way_acc', 0)),
            }
        metrics['model_a'] = ma

    cos_csv = os.path.join(reports_dir, 'model_a_cosine_retrieval_metrics.csv')
    if os.path.exists(cos_csv):
        if 'model_a' not in metrics:
            metrics['model_a'] = {}
        metrics['model_a']['cosine_similarity'] = {
            k: float(v) for k, v in pd.read_csv(cos_csv).iloc[0].items()
            if k != 'strategy'
        }

    gen_csv = os.path.join(reports_dir, 'model_a_text_generation_metrics.csv')
    if os.path.exists(gen_csv):
        metrics['text_generation'] = {
            k: float(v) for k, v in pd.read_csv(gen_csv).iloc[0].items()
        }

    mb_csv = os.path.join(reports_dir, 'model_b_metrics.csv')
    if os.path.exists(mb_csv):
        mb = {}
        for _, row in pd.read_csv(mb_csv).iterrows():
            model = str(row.get('model', ''))
            split = str(row.get('split', ''))
            key = f"{model}_{split}"
            mb[key] = {
                'accuracy':  float(row.get('accuracy', 0)),
                'f1':        float(row.get('f1', 0)),
                'precision': float(row.get('precision', 0)),
                'recall':    float(row.get('recall', 0)),
            }
        metrics['model_b'] = mb

    return metrics


if __name__ == '__main__':
    sample = (
        "The Amazon rainforest is often referred to as the lungs of the Earth. "
        "It produces 20 percent of the world's oxygen and is home to more than "
        "10 million species of plants, animals, and insects. The rainforest covers "
        "5.5 million square kilometres across nine countries. Deforestation is one "
        "of the biggest threats to this vital ecosystem. Scientists are urging "
        "governments and companies to take immediate action to protect the forest."
    )
    result = run_inference(sample)
    for i, q in enumerate(result['questions'], 1):
        print(f"\n--- Question {i}/5 ---")
        print(f"Q : {q['question']}")
        for k, v in q['options'].items():
            marker = " ← correct" if k == q['correct_label'] else ""
            print(f"  {k}: {v}{marker}")
    print(f"\nLatency: {result['latency_ms']} ms")


## TEST INFERENCE


In [ ]:
"""
test_inference.py
Unit tests for preprocessing utilities, TF-IDF cosine similarity, and
the inference pipeline.

Run with:  python -m pytest tests/test_inference.py -v
"""

import sys
import os

try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    _SRC_DIR = os.path.join(_THIS_DIR, '..', 'src')
except NameError:
    _SRC_DIR = os.path.join(os.getcwd(), 'src')
sys.path.insert(0, os.path.normpath(_SRC_DIR))

# from preprocessing import (
#     clean_text, tokenize, split_into_sentences,
#     cosine_similarity_feature, tfidf_cosine,
#     build_one_sample, build_tfidf_vectorizer,
# )
# from model_a_train import (
#     generate_questions_from_passage,
#     cosine_similarity_accuracy,
# )
# from inference import remove_answer_from_question, shuffle_options, run_inference


# ─────────────────────────────────────────────────────────────────────────────
# preprocessing — text cleaning
# ─────────────────────────────────────────────────────────────────────────────

def test_clean_text_removes_punctuation():
    result = clean_text("Hello, World! This is a test.")
    assert ',' not in result
    assert '!' not in result
    assert '.' not in result


def test_clean_text_lowercases():
    result = clean_text("UPPER lower MiXeD")
    assert result == result.lower()


def test_tokenize_removes_stopwords():
    tokens = tokenize("The cat sat on the mat")
    assert 'the' not in tokens
    assert 'on' not in tokens
    assert 'cat' in tokens or 'sat' in tokens or 'mat' in tokens


def test_tokenize_removes_short_words():
    tokens = tokenize("I am a big elephant")
    assert 'i' not in tokens
    assert 'a' not in tokens


def test_split_into_sentences_basic():
    text  = "First sentence. Second sentence. Third one!"
    sents = split_into_sentences(text)
    assert len(sents) >= 2


def test_build_one_sample_returns_string():
    result = build_one_sample("Article text here.", "What is this?", "An answer")
    assert isinstance(result, str)
    assert len(result) > 0


# ─────────────────────────────────────────────────────────────────────────────
# Jaccard / backward-compat cosine_similarity_feature
# ─────────────────────────────────────────────────────────────────────────────

def test_cosine_similarity_feature_identical():
    sim = cosine_similarity_feature("dogs are animals", "dogs are animals")
    assert sim > 0.8


def test_cosine_similarity_feature_no_overlap():
    sim = cosine_similarity_feature("dog cat fish", "apple orange mango")
    assert sim == 0.0


# ─────────────────────────────────────────────────────────────────────────────
# TRUE TF-IDF cosine similarity
# ─────────────────────────────────────────────────────────────────────────────

CORPUS = [
    "The Amazon rainforest covers 5.5 million square kilometres.",
    "Deforestation threatens the Amazon ecosystem and its biodiversity.",
    "The world's oxygen production relies partly on the Amazon.",
    "Scientists urge governments to protect the rainforest immediately.",
    "Millions of species live in the Amazon rainforest.",
]


def test_build_tfidf_vectorizer_returns_vectorizer():
    vec = build_tfidf_vectorizer(CORPUS)
    assert vec is not None
    # Should be able to transform a new text
    result = vec.transform(["Amazon rainforest"])
    assert result.shape[0] == 1


def test_tfidf_cosine_similarity_identical_texts():
    vec = build_tfidf_vectorizer(CORPUS)
    sim = tfidf_cosine("Amazon rainforest ecosystem", "Amazon rainforest ecosystem", vec)
    assert sim > 0.99, f"Expected > 0.99, got {sim}"


def test_tfidf_cosine_similarity_no_overlap():
    vec = build_tfidf_vectorizer(CORPUS)
    sim = tfidf_cosine("xyz foo bar", "abc def ghi", vec)
    assert sim == 0.0, f"Expected 0.0, got {sim}"


def test_tfidf_cosine_similarity_partial_overlap_in_range():
    vec = build_tfidf_vectorizer(CORPUS)
    sim = tfidf_cosine(
        "The Amazon rainforest is vital for oxygen production.",
        "Amazon produces oxygen for the world.",
        vec,
    )
    assert 0.0 <= sim <= 1.0, f"Similarity out of [0,1]: {sim}"


def test_tfidf_cosine_higher_for_similar_than_dissimilar():
    vec = build_tfidf_vectorizer(CORPUS)
    article = "The Amazon rainforest is the lungs of the Earth."
    sim_similar   = tfidf_cosine(article, "Amazon rainforest produces oxygen.", vec)
    sim_dissimilar = tfidf_cosine(article, "Football is popular in Brazil.", vec)
    assert sim_similar >= sim_dissimilar, (
        f"Expected similar ({sim_similar:.4f}) >= dissimilar ({sim_dissimilar:.4f})"
    )


# ─────────────────────────────────────────────────────────────────────────────
# cosine_similarity_accuracy (professor's metric)
# ─────────────────────────────────────────────────────────────────────────────

def test_cosine_similarity_accuracy_returns_dict():
    import pandas as pd
    vec = build_tfidf_vectorizer(CORPUS)
    # Build a tiny synthetic dataframe
    df = pd.DataFrame([{
        'article': "The Amazon rainforest covers millions of species and produces oxygen.",
        'question': "What does the Amazon produce?",
        'A': 'oxygen',
        'B': 'nothing',
        'C': 'pollution',
        'D': 'sand',
        'answer': 'A',
    }])
    result = cosine_similarity_accuracy(vec, df)
    assert 'accuracy' in result
    assert 'avg_correct_sim' in result
    assert 'avg_wrong_sim' in result
    assert 'sim_gap' in result
    assert 0.0 <= result['accuracy'] <= 1.0


def test_cosine_similarity_accuracy_perfect_case():
    """When correct option IS the article text, cosine should be 1.0 → accuracy 1.0."""
    import pandas as pd
    article = "The Amazon rainforest produces oxygen and hosts millions of species."
    vec     = build_tfidf_vectorizer([article] + CORPUS)
    df = pd.DataFrame([{
        'article': article,
        'question': "What does the Amazon do?",
        'A': article,        # correct — identical to article → highest cosine
        'B': 'xyz abc def',
        'C': 'foo bar baz',
        'D': 'random words',
        'answer': 'A',
    }])
    result = cosine_similarity_accuracy(vec, df)
    assert result['accuracy'] == 1.0, f"Expected 1.0, got {result['accuracy']}"


# ─────────────────────────────────────────────────────────────────────────────
# question generation
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_ARTICLE = (
    "The Amazon rainforest covers 5.5 million square kilometres across nine countries. "
    "It produces about 20 percent of the world's oxygen and is home to millions of species. "
    "Deforestation is one of the biggest threats to this ecosystem."
)


def test_generate_questions_returns_list():
    qs = generate_questions_from_passage(SAMPLE_ARTICLE, count=3)
    assert isinstance(qs, list)
    assert len(qs) > 0


def test_each_question_is_dict():
    qs = generate_questions_from_passage(SAMPLE_ARTICLE, count=2)
    for item in qs:
        assert isinstance(item, dict)
        assert 'question' in item and 'answer' in item


def test_answer_not_identical_to_question():
    qs = generate_questions_from_passage(SAMPLE_ARTICLE, count=3)
    ans = qs[0].get('answer', '')
    assert isinstance(ans, str) and len(ans) > 0


# ─────────────────────────────────────────────────────────────────────────────
# answer removal (masking utility)
# ─────────────────────────────────────────────────────────────────────────────

def test_remove_answer_from_question_masks_token():
    q      = "What does the passage say about amazon rainforest coverage?"
    answer = "amazon rainforest"
    result = remove_answer_from_question(q, answer)
    assert 'amazon' not in result.lower() or '___' in result


def test_remove_answer_short_tokens_ignored():
    q      = "What is it about?"
    answer = "it"
    result = remove_answer_from_question(q, answer)
    # Short tokens (len <= 3) should NOT be masked
    assert result == q


# ─────────────────────────────────────────────────────────────────────────────
# shuffle_options
# ─────────────────────────────────────────────────────────────────────────────

def test_shuffle_options_all_labels_present():
    opts, label = shuffle_options("correct", ["wrong1", "wrong2", "wrong3"])
    assert set(opts.keys()) == {'A', 'B', 'C', 'D'}


def test_shuffle_options_correct_answer_in_dict():
    opts, label = shuffle_options("correct", ["wrong1", "wrong2", "wrong3"])
    assert opts[label] == "correct"


def test_shuffle_options_correct_label_valid():
    _, label = shuffle_options("correct", ["w1", "w2", "w3"])
    assert label in ['A', 'B', 'C', 'D']


def test_shuffle_options_all_four_options_filled():
    opts, _ = shuffle_options("answer", ["d1", "d2", "d3"])
    assert len(opts) == 4
    assert all(v != '' for v in opts.values())


# ─────────────────────────────────────────────────────────────────────────────
# run_inference (no race_rows — template-based path)
# ─────────────────────────────────────────────────────────────────────────────

def test_run_inference_returns_dict():
    result = run_inference(SAMPLE_ARTICLE)
    assert isinstance(result, dict)
    assert 'questions' in result
    assert 'latency_ms' in result


def test_run_inference_questions_count():
    result = run_inference(SAMPLE_ARTICLE)
    assert 5 <= len(result['questions']) <= 10


def test_run_inference_question_structure():
    result = run_inference(SAMPLE_ARTICLE)
    for q in result['questions']:
        assert 'question' in q
        assert 'options' in q
        assert 'correct_label' in q
        assert 'correct_answer' in q
        assert 'hints' in q
        assert set(q['options'].keys()) == {'A', 'B', 'C', 'D'}
        assert q['correct_label'] in ['A', 'B', 'C', 'D']


def test_run_inference_correct_answer_in_options():
    result = run_inference(SAMPLE_ARTICLE)
    for q in result['questions']:
        label = q['correct_label']
        assert q['options'][label] == q['correct_answer']


def test_run_inference_latency_is_positive():
    result = run_inference(SAMPLE_ARTICLE)
    assert result['latency_ms'] >= 0


# ─────────────────────────────────────────────────────────────────────────────
# run_inference with race_rows (real RACE dataset path)
# ─────────────────────────────────────────────────────────────────────────────

RACE_ROW_SAMPLE = {
    'article':  SAMPLE_ARTICLE,
    'question': "How many countries does the Amazon rainforest span?",
    'A': 'Nine',
    'B': 'Seven',
    'C': 'Twelve',
    'D': 'Five',
    'answer':   'A',
    'id':       'test_001',
}


def test_run_inference_with_race_rows():
    result = run_inference(SAMPLE_ARTICLE, race_rows=[RACE_ROW_SAMPLE])
    assert isinstance(result, dict)
    assert len(result['questions']) >= 1


def test_run_inference_race_row_uses_real_question():
    result = run_inference(SAMPLE_ARTICLE, race_rows=[RACE_ROW_SAMPLE])
    first_q = result['questions'][0]
    assert first_q['question'] == RACE_ROW_SAMPLE['question']


def test_run_inference_race_row_correct_label():
    result = run_inference(SAMPLE_ARTICLE, race_rows=[RACE_ROW_SAMPLE])
    first_q = result['questions'][0]
    assert first_q['correct_label'] == 'A'
    assert first_q['correct_answer'] == 'Nine'


def test_run_inference_race_rows_fills_up_to_five():
    """When fewer than 5 race_rows given, template questions fill remaining slots."""
    result = run_inference(SAMPLE_ARTICLE, race_rows=[RACE_ROW_SAMPLE])
    assert len(result['questions']) <= 5


def test_run_inference_multiple_race_rows():
    row2 = {
        'article':  SAMPLE_ARTICLE,
        'question': "What percentage of world oxygen does the Amazon produce?",
        'A': '10 percent',
        'B': '20 percent',
        'C': '30 percent',
        'D': '50 percent',
        'answer':   'B',
        'id':       'test_001',
    }
    result = run_inference(SAMPLE_ARTICLE, race_rows=[RACE_ROW_SAMPLE, row2])
    assert len(result['questions']) >= 2
    assert result['questions'][1]['correct_label'] == 'B'
    assert result['questions'][1]['correct_answer'] == '20 percent'


# ─────────────────────────────────────────────────────────────────────────────
# Manual runner (no pytest needed)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    import traceback
    tests = [
        test_clean_text_removes_punctuation,
        test_clean_text_lowercases,
        test_tokenize_removes_stopwords,
        test_tokenize_removes_short_words,
        test_split_into_sentences_basic,
        test_build_one_sample_returns_string,
        test_cosine_similarity_feature_identical,
        test_cosine_similarity_feature_no_overlap,
        test_build_tfidf_vectorizer_returns_vectorizer,
        test_tfidf_cosine_similarity_identical_texts,
        test_tfidf_cosine_similarity_no_overlap,
        test_tfidf_cosine_similarity_partial_overlap_in_range,
        test_tfidf_cosine_higher_for_similar_than_dissimilar,
        test_cosine_similarity_accuracy_returns_dict,
        test_cosine_similarity_accuracy_perfect_case,
        test_generate_questions_returns_list,
        test_each_question_is_dict,
        test_answer_not_identical_to_question,
        test_remove_answer_from_question_masks_token,
        test_remove_answer_short_tokens_ignored,
        test_shuffle_options_all_labels_present,
        test_shuffle_options_correct_answer_in_dict,
        test_shuffle_options_correct_label_valid,
        test_shuffle_options_all_four_options_filled,
        test_run_inference_returns_dict,
        test_run_inference_questions_count,
        test_run_inference_question_structure,
        test_run_inference_correct_answer_in_options,
        test_run_inference_latency_is_positive,
        test_run_inference_with_race_rows,
        test_run_inference_race_row_uses_real_question,
        test_run_inference_race_row_correct_label,
        test_run_inference_race_rows_fills_up_to_five,
        test_run_inference_multiple_race_rows,
    ]
    passed, failed = 0, 0
    for t in tests:
        try:
            t()
            print(f"  PASS  {t.__name__}")
            passed += 1
        except Exception as e:
            print(f"  FAIL  {t.__name__}: {e}")
            traceback.print_exc()
            failed += 1
    print(f"\n{'-'*50}")
    print(f"  {passed} passed, {failed} failed")


In [ ]:
# from src.inference import run_inference
result = run_inference(
    "Photosynthesis is the process by which plants use sunlight, "
    "water and carbon dioxide to produce oxygen and energy in the form of sugar."
)
for q in result["questions"]:
    print(f"Q: {q['question']}")
    print(f"A: {q['correct']}")
    print(f"D: {q['distractors']}")
    print(f"H: {q['hints']}")
    print()
print(f"Latency: {result['latency_ms']:.0f} ms")

In [ ]:
"""
evaluate.py
──────────────────────────────────────────────────────────────────────────────
Full evaluation of Model A and Model B on the test split.

PRIMARY METRICS (NLP generation task — professor's requirement):
  BLEU   — n-gram precision between predicted answer and gold answer
  ROUGE  — recall-oriented overlap (ROUGE-1, ROUGE-2, ROUGE-L)
  METEOR — alignment-based metric covering synonyms and stemming

SECONDARY METRICS:
  Cosine Similarity Accuracy — TF-IDF cosine between article and options
  Binary classification: Accuracy, Precision, Recall, F1, Confusion Matrix
  4-way MCQ accuracy (ML model picks best option from OHE features)
  Train↔Test domain similarity

Usage:
  python src/evaluate.py
"""

import os
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib  # type: ignore[reportMissingModuleSource]
matplotlib.use('Agg')
import matplotlib.pyplot as plt  # type: ignore[reportMissingModuleSource]
import seaborn as sns  # type: ignore[reportMissingModuleSource]
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
)
from scipy.sparse import load_npz
import joblib

try:
    _THIS_DIR = os.path.dirname(os.path.abspath(__file__))
    PROJECT_ROOT = os.path.abspath(os.path.join(_THIS_DIR, '..'))
except NameError:
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import PROCESSED_DIR, BASE_DIR
from src.model_a_train import (
    compute_4way_accuracy,
    cosine_similarity_accuracy,
    compute_train_test_domain_similarity,
    compute_generation_metrics as ma_compute_generation_metrics,
    generate_questions_from_passage,
)
from src.nn_models import AnswerVerifier, load_checkpoint

MODEL_A_DIR = os.path.join(BASE_DIR, 'models', 'model_a', 'neural')
MODEL_B_DIR = os.path.join(BASE_DIR, 'models', 'model_b', 'neural')
REPORTS_DIR = os.path.join(PROCESSED_DIR, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def print_section(title):
    print(f"\n{'═' * 60}")
    print(f"  {title}")
    print('═' * 60)


def print_subsection(title):
    print(f"\n  {'─' * 56}")
    print(f"    {title}")
    print(f"  {'─' * 56}")


def _ensure_nltk_resources():
    """Download required NLTK data if not already present."""
    import nltk
    resources = [
        ('tokenizers/punkt',     'punkt'),
        ('tokenizers/punkt_tab', 'punkt_tab'),
        ('corpora/wordnet',      'wordnet'),
        ('corpora/omw-1.4',      'omw-1.4'),
    ]
    for path, name in resources:
        try:
            nltk.data.find(path)
        except LookupError:
            try:
                nltk.download(name, quiet=True)
            except Exception:
                pass


def _score_generation_pair(ref_text, hyp_text, rouge_scorer_obj, smoother):
    import nltk
    from nltk.translate.bleu_score import sentence_bleu
    from nltk.translate.meteor_score import meteor_score

    ref_tokens = nltk.word_tokenize(str(ref_text).lower())
    hyp_tokens = nltk.word_tokenize(str(hyp_text).lower())
    if not ref_tokens or not hyp_tokens:
        return None

    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)
    try:
        met = meteor_score([ref_tokens], hyp_tokens)
    except Exception:
        met = 0.0
    rouge_out = rouge_scorer_obj.score(str(ref_text), str(hyp_text))

    return {
        'bleu': bleu,
        'meteor': met,
        'rouge1_f': rouge_out['rouge1'].fmeasure,
        'rouge1_p': rouge_out['rouge1'].precision,
        'rouge1_r': rouge_out['rouge1'].recall,
        'rouge2_f': rouge_out['rouge2'].fmeasure,
        'rouge2_p': rouge_out['rouge2'].precision,
        'rouge2_r': rouge_out['rouge2'].recall,
        'rougeL_f': rouge_out['rougeL'].fmeasure,
    }


# ─────────────────────────────────────────────────────────────────────────────
# PRIMARY METRICS — BLEU / ROUGE / METEOR
# ─────────────────────────────────────────────────────────────────────────────

def compute_generation_metrics(test_df, tfidf_vec=None, sample_n=300):
    """
    Generation metrics via model_a_train (BLEU / ROUGE / METEOR on generated answers).
    """
    print_section("PRIMARY METRICS — BLEU / ROUGE / METEOR")
    print("  Comparing generated answers (passage pipeline) vs. RACE references …")

    sample_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)
    all_generated = []
    for _, row in sample_df.iterrows():
        article = str(row.get('article', ''))
        if len(article) < 50:
            continue
        all_generated.extend(generate_questions_from_passage(article, count=2))

    if not all_generated:
        print("  WARNING: no generated samples.")
        return {}

    # model_a_train.generation_metrics expects the keyword `n_sample`
    results = ma_compute_generation_metrics(all_generated, sample_df, n_sample=sample_n)
    if not results:
        return {}

    n = int(results.get('n_samples', 0))
    print(f"\n  Evaluated {n} samples (test split)")
    print(f"\n  ┌{'─'*40}┐")
    print(f"  │ {'METRIC':<28}  {'SCORE':>8} │")
    print(f"  ├{'─'*40}┤")
    print(f"  │ {'BLEU':<28}  {results.get('bleu', 0):>8.4f} │")
    print(f"  │ {'METEOR':<28}  {results.get('meteor', 0):>8.4f} │")
    print(f"  │ {'ROUGE-1  F1':<28}  {results.get('rouge1_f', 0):>8.4f} │")
    print(f"  │ {'ROUGE-2  F1':<28}  {results.get('rouge2_f', 0):>8.4f} │")
    print(f"  │ {'ROUGE-L  F1':<28}  {results.get('rougeL_f', 0):>8.4f} │")
    print(f"  └{'─'*40}┘")

    plot_payload = {
        'bleu': results.get('bleu', 0),
        'meteor': results.get('meteor', 0),
        'rouge1_f': results.get('rouge1_f', 0),
        'rouge2_f': results.get('rouge2_f', 0),
        'rougeL_f': results.get('rougeL_f', 0),
    }
    _save_generation_metrics_plot(plot_payload)
    return results


def compute_question_generation_metrics(test_df, sample_n=300, n_candidates=5):
    """
    Evaluate generated questions vs. gold questions using BLEU / ROUGE / METEOR.

    For each test sample:
      Candidates: generate_questions_from_passage(article)
      Reference : gold question from RACE dataset
      Selection : use the best-matching candidate by ROUGE-L F1
    """
    print_section("PRIMARY METRICS — BLEU / ROUGE / METEOR (Questions)")
    print("  Comparing generated questions vs. RACE gold questions …")

    _ensure_nltk_resources()

    from nltk.translate.bleu_score import SmoothingFunction
    from rouge_score import rouge_scorer as rs_lib

    rouge_scorer_obj = rs_lib.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
    )
    smoother = SmoothingFunction().method1

    sample_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)

    bleu_scores, meteor_scores = [], []
    rouge1_f, rouge2_f, rougeL_f = [], [], []
    rouge1_p, rouge1_r = [], []
    rouge2_p, rouge2_r = [], []

    for _, row in sample_df.iterrows():
        article = str(row['article'])
        gold_q  = str(row['question'])

        # `generate_questions_from_passage` (alias of `compose_questions`) expects
        # the `count` keyword for number of questions.
        candidates_rows = generate_questions_from_passage(
            article, count=n_candidates
        )
        candidates = [d['question'] for d in candidates_rows]
        if not candidates:
            continue

        # Pick best candidate by ROUGE-L F1 to avoid penalizing diverse phrasing
        best = None
        best_rl = -1.0
        for cand in candidates:
            scores = _score_generation_pair(gold_q, cand, rouge_scorer_obj, smoother)
            if not scores:
                continue
            if scores['rougeL_f'] > best_rl:
                best_rl = scores['rougeL_f']
                best = scores

        if not best:
            continue

        bleu_scores.append(best['bleu'])
        meteor_scores.append(best['meteor'])
        rouge1_f.append(best['rouge1_f'])
        rouge1_p.append(best['rouge1_p'])
        rouge1_r.append(best['rouge1_r'])
        rouge2_f.append(best['rouge2_f'])
        rouge2_p.append(best['rouge2_p'])
        rouge2_r.append(best['rouge2_r'])
        rougeL_f.append(best['rougeL_f'])

    n = len(bleu_scores)
    if n == 0:
        print("  WARNING: no valid samples processed.")
        return {}

    results = {
        'q_bleu':      float(np.mean(bleu_scores)),
        'q_meteor':    float(np.mean(meteor_scores)),
        'q_rouge1_f':  float(np.mean(rouge1_f)),
        'q_rouge1_p':  float(np.mean(rouge1_p)),
        'q_rouge1_r':  float(np.mean(rouge1_r)),
        'q_rouge2_f':  float(np.mean(rouge2_f)),
        'q_rouge2_p':  float(np.mean(rouge2_p)),
        'q_rouge2_r':  float(np.mean(rouge2_r)),
        'q_rougeL_f':  float(np.mean(rougeL_f)),
        'n_samples':    n,
        'n_candidates': n_candidates,
    }

    print(f"\n  Evaluated {n} samples (test split)")
    print(f"\n  ┌{'─'*40}┐")
    print(f"  │ {'METRIC':<28}  {'SCORE':>8} │")
    print(f"  ├{'─'*40}┤")
    print(f"  │ {'BLEU':<28}  {results['q_bleu']:>8.4f} │")
    print(f"  │ {'METEOR':<28}  {results['q_meteor']:>8.4f} │")
    print(f"  │ {'ROUGE-1  F1':<28}  {results['q_rouge1_f']:>8.4f} │")
    print(f"  │   {'Precision':<26}  {results['q_rouge1_p']:>8.4f} │")
    print(f"  │   {'Recall':<26}  {results['q_rouge1_r']:>8.4f} │")
    print(f"  │ {'ROUGE-2  F1':<28}  {results['q_rouge2_f']:>8.4f} │")
    print(f"  │ {'ROUGE-L  F1':<28}  {results['q_rougeL_f']:>8.4f} │")
    print(f"  └{'─'*40}┘")

    return results


def _save_generation_metrics_plot(gen_metrics):
    """Bar chart for BLEU / ROUGE / METEOR scores."""
    labels = ['BLEU', 'METEOR', 'ROUGE-1\nF1', 'ROUGE-2\nF1', 'ROUGE-L\nF1']
    values = [
        gen_metrics.get('bleu', 0),
        gen_metrics.get('meteor', 0),
        gen_metrics.get('rouge1_f', 0),
        gen_metrics.get('rouge2_f', 0),
        gen_metrics.get('rougeL_f', 0),
    ]
    colors = ['#D72638', '#a0172a', '#28a745', '#F59E0B', '#6B7280']

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(labels, values, color=colors, alpha=0.88, width=0.55)
    ax.set_ylim(0, max(max(values) * 1.4, 0.1))
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('Generation Evaluation — BLEU / ROUGE / METEOR',
                 fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='#6B7280', linewidth=0.8)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    plt.tight_layout()
    out = os.path.join(REPORTS_DIR, 'generation_metrics_plot.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"\n  Plot saved → {out}")


# ─────────────────────────────────────────────────────────────────────────────
# COSINE SIMILARITY ACCURACY
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_cosine_similarity(tfidf_vec, test_df, train_df=None, sample_n=500):
    """TF-IDF cosine similarity accuracy — argmax(cosine) predicts correct option."""
    print_section("COSINE SIMILARITY ACCURACY")

    eval_df = test_df.sample(min(sample_n, len(test_df)), random_state=42)
    cos = cosine_similarity_accuracy(tfidf_vec, eval_df)

    gap_flag = "correct > wrong ✓" if cos['sim_gap'] > 0 else "inverted ✗"
    print(f"\n  {'Cosine Similarity Accuracy':<35}: {cos['accuracy']:.4f}  ({cos['accuracy']*100:.1f}%)")
    print(f"  {'Avg similarity — correct option':<35}: {cos['avg_correct_sim']:.4f}")
    print(f"  {'Avg similarity — wrong options':<35}: {cos['avg_wrong_sim']:.4f}")
    print(f"  {'Similarity gap (correct − wrong)':<35}: {cos['sim_gap']:.4f}  [{gap_flag}]")

    if train_df is not None:
        # model_a_train.domain_overlap expects `n_sample` keyword
        domain_sim = compute_train_test_domain_similarity(
            train_df, test_df, tfidf_vec, n_sample=200
        )
        print(f"\n  Train↔Test domain similarity: {domain_sim:.4f}")
        cos['domain_similarity'] = domain_sim

    _save_cosine_sim_plot(tfidf_vec, eval_df)
    return cos


def _save_cosine_sim_plot(tfidf_vec, eval_df):
    from src.preprocessing import tfidf_cosine

    correct_sims, wrong_sims = [], []
    for _, row in eval_df.iterrows():
        article = str(row['article'])
        gold    = str(row['answer']).strip().upper()
        for opt in ['A', 'B', 'C', 'D']:
            sim = tfidf_cosine(article, str(row[opt]), tfidf_vec)
            (correct_sims if opt == gold else wrong_sims).append(sim)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    fig.suptitle('TF-IDF Cosine Similarity: Correct vs Wrong Options',
                 fontsize=13, fontweight='bold')
    axes[0].hist(correct_sims, bins=30, alpha=0.75, color='#28a745', label='Correct')
    axes[0].hist(wrong_sims,   bins=30, alpha=0.75, color='#D72638', label='Wrong')
    axes[0].axvline(np.mean(correct_sims), color='#28a745', lw=2, ls='--')
    axes[0].axvline(np.mean(wrong_sims),   color='#D72638', lw=2, ls='--')
    axes[0].set_xlabel('Cosine Similarity'); axes[0].set_ylabel('Count')
    axes[0].legend(); axes[0].set_title('Distribution')
    means = [np.mean(correct_sims), np.mean(wrong_sims)]
    stds  = [np.std(correct_sims),  np.std(wrong_sims)]
    bars  = axes[1].bar(['Correct', 'Wrong'], means, yerr=stds,
                        color=['#28a745', '#D72638'], alpha=0.85,
                        capsize=8, error_kw={'linewidth': 2})
    axes[1].set_ylabel('Mean Cosine Similarity'); axes[1].set_title('Mean ± Std')
    for bar, m in zip(bars, means):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                     f'{m:.4f}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_DIR, 'cosine_sim_plot.png'), dpi=120, bbox_inches='tight')
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# MODEL A  (binary + 4-way MCQ)
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model_a(ohe_vec, test_df, tfidf_vec=None):
    print_section("MODEL A — BINARY CLASSIFICATION & 4-WAY MCQ")

    X_te_path = os.path.join(PROCESSED_DIR, 'X_test_ohe.npz')
    y_te_path  = os.path.join(PROCESSED_DIR, 'y_test.npy')
    results = {}

    # Load NN checkpoint
    nn_path = os.path.join(MODEL_A_DIR, 'answer_verifier.pt')
    if not os.path.exists(nn_path):
        print("  WARNING: NN checkpoint not found. Skipping Model A evaluation.")
        return results

    model = load_checkpoint(AnswerVerifier, nn_path)

    if os.path.exists(X_te_path) and os.path.exists(y_te_path):
        print_subsection("Binary classification (is option correct?)")
        X_te = load_npz(X_te_path)
        y_te = np.load(y_te_path)
        preds = model.predict(X_te)
        acc = accuracy_score(y_te, preds)
        p   = precision_score(y_te, preds, average='macro', zero_division=0)
        r   = recall_score(y_te,    preds, average='macro', zero_division=0)
        f1  = f1_score(y_te,        preds, average='macro', zero_division=0)
        cm  = confusion_matrix(y_te, preds)
        print(f"    {'NN':<30}: Acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f1:.4f}")
        # Save confusion matrix plot
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Incorrect', 'Correct'],
                    yticklabels=['Incorrect', 'Correct'], ax=ax)
        ax.set_title('NN — Confusion Matrix')
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
        plt.tight_layout()
        cm_path = os.path.join(REPORTS_DIR, 'nn_cm.png')
        plt.savefig(cm_path, dpi=100); plt.close()
        results['nn'] = {'accuracy': acc, 'precision': p, 'recall': r,
                         'f1': f1, 'confusion_matrix': cm}

    print_subsection("4-way MCQ accuracy")
    acc_4w = compute_4way_accuracy(model, ohe_vec, test_df)
    print(f"    {'NN':<30}: {acc_4w:.4f}  ({acc_4w*100:.1f}%)")
    if 'nn' not in results:
        results['nn'] = {}
    results['nn']['4way_acc'] = acc_4w

    return results


# ─────────────────────────────────────────────────────────────────────────────
# MODEL B
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_model_b():
    print_section("MODEL B — DISTRACTOR RANKER & HINT SCORER")
    path = os.path.join(MODEL_B_DIR, 'metrics.pkl')
    if not os.path.exists(path):
        print("  Not found. Run model_b_train.py first.")
        return {}
    metrics = joblib.load(path)
    d = metrics.get('distractor', {})
    h = metrics.get('hint', {})
    print(f"  Distractor Ranker: Acc={d.get('acc', 0):.4f}  F1={d.get('f1', 0):.4f}")
    print(f"  Hint Scorer:       Acc={h.get('acc', 0):.4f}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY TABLE
# ─────────────────────────────────────────────────────────────────────────────

def print_summary(gen_results, cos_results, ma_results):
    print_section("EVALUATION SUMMARY")
    print(f"\n  ┌{'─'*52}┐")
    print(f"  │ {'METRIC':<40}  {'VALUE':>8} │")
    print(f"  ├{'─'*52}┤")
    if gen_results:
        for key, lbl in [('bleu','★ BLEU (answer extraction)'),
                          ('meteor','★ METEOR'),
                          ('rouge1_f','★ ROUGE-1 F1'),
                          ('rouge2_f','★ ROUGE-2 F1'),
                          ('rougeL_f','★ ROUGE-L F1')]:
            print(f"  │ {lbl:<40}  {gen_results.get(key,0):>8.4f} │")
        for key, lbl in [('q_bleu','★ BLEU (question gen)'),
                          ('q_meteor','★ METEOR (question gen)'),
                          ('q_rouge1_f','★ ROUGE-1 F1 (question gen)'),
                          ('q_rouge2_f','★ ROUGE-2 F1 (question gen)'),
                          ('q_rougeL_f','★ ROUGE-L F1 (question gen)')]:
            if key in gen_results:
                print(f"  │ {lbl:<40}  {gen_results.get(key,0):>8.4f} │")
        print(f"  ├{'─'*52}┤")
    if cos_results:
        print(f"  │ {'Cosine Similarity Accuracy':<40}  {cos_results.get('accuracy',0):>8.4f} │")
        print(f"  │ {'Similarity Gap (correct−wrong)':<40}  {cos_results.get('sim_gap',0):>8.4f} │")
        print(f"  ├{'─'*52}┤")
    for key, lbl in [('nn', 'NN binary accuracy')]:
        v = ma_results.get(key, {}).get('accuracy')
        if v is not None:
            print(f"  │ {lbl:<40}  {v:>8.4f} │")
    print(f"  └{'─'*52}┘")


# ─────────────────────────────────────────────────────────────────────────────
# FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_full_evaluation():
    print_section("RACE RC PROJECT — FULL EVALUATION")

    ohe_path   = os.path.join(PROCESSED_DIR, 'ohe_vectorizer.pkl')
    tfidf_path = os.path.join(PROCESSED_DIR, 'tfidf_vectorizer.pkl')
    test_csv   = os.path.join(PROCESSED_DIR, 'test_clean.csv')
    train_csv  = os.path.join(PROCESSED_DIR, 'train_clean.csv')

    missing = [p for p in [ohe_path, tfidf_path, test_csv] if not os.path.exists(p)]
    if missing:
        print(f"\n  ERROR: Missing files:\n    " + '\n    '.join(missing))
        print("  Run preprocessing.py first.")
        return {}

    with open(ohe_path,   'rb') as f: ohe_vec   = pickle.load(f)
    with open(tfidf_path, 'rb') as f: tfidf_vec = pickle.load(f)

    test_df  = pd.read_csv(test_csv)
    train_df = pd.read_csv(train_csv) if os.path.exists(train_csv) else None

    # ── PRIMARY: BLEU / ROUGE / METEOR ────────────────────────────────────
    gen_results = {}
    try:
        gen_results = compute_generation_metrics(test_df, tfidf_vec, sample_n=300)
        q_results = compute_question_generation_metrics(test_df, sample_n=300)
        if q_results:
            gen_results.update(q_results)
        gen_out = os.path.join(REPORTS_DIR, 'generation_metrics.pkl')
        joblib.dump(gen_results, gen_out)
        print(f"  Generation metrics saved → {gen_out}")
    except ImportError as e:
        print(f"\n  WARNING: Could not compute generation metrics ({e}).")
        print("  Install: pip install nltk rouge-score")

    # ── COSINE SIMILARITY ACCURACY ─────────────────────────────────────────
    cos_results = evaluate_cosine_similarity(
        tfidf_vec, test_df, train_df=train_df, sample_n=500
    )

    # ── MODEL A (binary + 4-way) ───────────────────────────────────────────
    ma_results = evaluate_model_a(ohe_vec, test_df, tfidf_vec=tfidf_vec)

    # ── MODEL B ───────────────────────────────────────────────────────────
    mb_results = evaluate_model_b()

    # ── SUMMARY ───────────────────────────────────────────────────────────
    print_summary(gen_results, cos_results, ma_results)

    all_metrics = {
        'generation':       gen_results,
        'cosine_similarity': cos_results,
        'model_a':          ma_results,
        'model_b':          mb_results,
    }
    out_path = os.path.join(REPORTS_DIR, 'all_metrics.pkl')
    joblib.dump(all_metrics, out_path)
    print(f"\n  All metrics saved → {out_path}")
    print_section("Evaluation complete!")
    return all_metrics


if __name__ == '__main__':
    run_full_evaluation()
